# Parking Enforcement Intelligence

This notebook builds an AI-driven parking hotspot pipeline from the raw violation CSV. It produces two submission artifacts: `outputs\final_hotspots.csv` and `outputs\parking_hotspot_map.html`.

Run cells top to bottom. Keep the CSV in the same folder as this notebook, or update `DATA_FILENAME` in Cell 3.

Required packages: pandas, numpy, scipy, scikit-learn, folium. Optional package: h3 for true hex cells; the code falls back to a grid if h3 is unavailable.


In [46]:
# CELL 0: Dependency check

from importlib.util import find_spec

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "folium": "folium",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if find_spec(import_name) is None]
if missing:
    raise ImportError(
        "Missing required packages: " + ", ".join(missing) +
        ". Install them with: pip install " + " ".join(missing)
    )

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        def display(obj):
            print(obj)

print("Dependency check passed.")
print("Optional h3 installed:", find_spec("h3") is not None)


Dependency check passed.
Optional h3 installed: True


In [3]:
import json
import ast
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# =========================
# DATA CONFIG
# =========================

DATA_FILENAME = "jan to may police violation_anonymized791b166.csv"

DATA_PATH_CANDIDATES = [
    Path(DATA_FILENAME),
    Path("data") / DATA_FILENAME,
    Path.cwd() / DATA_FILENAME,
    Path.cwd() / "data" / DATA_FILENAME,
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the input CSV. Place it next to this notebook or inside a data/ folder.\n"
        f"Expected filename: {DATA_FILENAME}"
    )

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FINAL_HOTSPOTS_PATH = OUTPUT_DIR / "final_hotspots.csv"
MAP_HTML_PATH = OUTPUT_DIR / "parking_hotspot_map.html"

# =========================
# GLOBAL PIPELINE CONFIG
# =========================

BBOX = {
    "lat_min": 12.70,
    "lat_max": 13.25,
    "lon_min": 77.30,
    "lon_max": 77.85,
}

APPROVED_ONLY = False
BAD_STATUSES = ["rejected", "duplicate"]

DROP_COLS = [
    "description",
    "closed_datetime",
    "action_taken_timestamp",
    "data_sent_to_scita_timestamp",
]

SPATIAL_MODE = "junction_plus_h3"
H3_RESOLUTION = 9
GRID_SIZE_DEGREES = 0.002

TARGET_COL = "daily_impact_signal"
FORECAST_HORIZON_DAYS = 7

ROLL_WINDOWS = [7, 14, 30]
LAG_DAYS = [1, 7, 14]

VALIDATION_DATES = [
    "2024-01-01",
    "2024-02-01",
    "2024-03-01",
    "2024-04-01",
]

TOP_K_LIST = [10, 25, 50, 100]

MIN_HIST_EVENTS_FOR_EVAL = 5
MIN_HIST_EVENTS_FOR_TRAIN = 5
MIN_ROWS_TO_TRAIN_FOLD = 200

MODEL_TYPE = "hist_gbdt"
WEIGHT_STEP = 0.05

NEIGHBOR_RADIUS_KM = 0.5
EARTH_RADIUS_KM = 6371.0

print("Data path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR.resolve())
print("Random state:", RANDOM_STATE)

Data path: jan to may police violation_anonymized791b166.csv
Output directory: C:\Users\Harshita\Downloads\Flipkart_hackathon\Phase_2\outputs
Random state: 42


In [4]:
# =========================
# LOAD RAW DATA
# =========================

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()

print("Raw shape:", df.shape)

required_cols = [
    "latitude",
    "longitude",
    "validation_status",
    "created_datetime",
    "violation_type",
    "offence_code",
    "vehicle_type",
    "updated_vehicle_type",
    "junction_name",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# =========================
# BASIC CLEANING
# =========================

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

df = df[
    df["latitude"].between(BBOX["lat_min"], BBOX["lat_max"])
    & df["longitude"].between(BBOX["lon_min"], BBOX["lon_max"])
].copy()

df["validation_status"] = (
    df["validation_status"]
    .astype("string")
    .str.lower()
    .str.strip()
)

bad_mask = df["validation_status"].isin(BAD_STATUSES).fillna(False)
df = df[~bad_mask].copy()

if APPROVED_ONLY:
    df = df[df["validation_status"].eq("approved").fillna(False)].copy()

df = df.reset_index(drop=True)

# =========================
# TRUST SCORE
# =========================
# Approved records get full confidence.
# Pending / processing / missing records are retained but down-weighted.

TRUST_MAP = {
    "approved": 1.00,
    "processing": 0.60,
    "created1": 0.60,
}

df["trust_score"] = df["validation_status"].map(TRUST_MAP).fillna(0.50)

# =========================
# TIME FEATURES
# =========================

df["created_datetime"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
df = df[df["created_datetime"].notna()].copy()

df["created_ist"] = df["created_datetime"].dt.tz_convert("Asia/Kolkata")
df["created_ist_naive"] = df["created_ist"].dt.tz_localize(None)

df["date"] = df["created_ist_naive"].dt.normalize()
df["hour"] = df["created_ist_naive"].dt.hour
df["dayofweek"] = df["created_ist_naive"].dt.dayofweek
df["month"] = df["created_ist_naive"].dt.month
df["year_month"] = df["created_ist_naive"].dt.to_period("M").astype(str)

df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
df["time_bucket_3h"] = (df["hour"] // 3) * 3
df["is_peak_hour"] = df["hour"].isin([8, 9, 10, 17, 18, 19, 20]).astype(int)

# =========================
# LIST-LIKE COLUMN PARSING
# =========================

def parse_list_cell(x):
    if isinstance(x, list):
        values = x
    elif pd.isna(x):
        values = []
    else:
        text = str(x).strip()

        try:
            parsed = json.loads(text)
            values = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            try:
                parsed = ast.literal_eval(text)
                values = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                values = [text]

    return [str(v).upper().strip() for v in values if pd.notna(v) and str(v).strip()]

df["violation_list"] = df["violation_type"].apply(parse_list_cell)
df["offence_code_list"] = df["offence_code"].apply(parse_list_cell)
df["n_violation_types"] = df["violation_list"].apply(len)

# =========================
# VEHICLE AND JUNCTION FIELDS
# =========================

df["vehicle_type_final"] = df["updated_vehicle_type"].where(
    df["updated_vehicle_type"].notna(),
    df["vehicle_type"],
)

df["vehicle_type_final"] = (
    df["vehicle_type_final"]
    .astype("string")
    .str.upper()
    .str.strip()
)

df["junction_name"] = df["junction_name"].fillna("No Junction")
df["is_camera_junction"] = df["junction_name"].astype(str).str.startswith("BTP").astype(int)

# =========================
# SANITY CHECKS
# =========================

print("Clean shape:", df.shape)
print("Date range:", df["created_ist_naive"].min(), "to", df["created_ist_naive"].max())

print("\nValidation status distribution:")
display(df["validation_status"].value_counts(dropna=False).to_frame("count"))

print("\nTrust score distribution:")
display(df["trust_score"].value_counts(dropna=False).sort_index().to_frame("count"))

print("\nTop violation labels:")
violation_counter = Counter(v for row in df["violation_list"] for v in row)
display(pd.DataFrame(violation_counter.most_common(15), columns=["violation", "count"]))

print("\nTop vehicle types:")
display(df["vehicle_type_final"].value_counts(dropna=False).head(15).to_frame("count"))

df.head()

Raw shape: (298450, 24)
Clean shape: (248357, 36)
Date range: 2023-11-10 00:41:46 to 2024-04-08 23:00:46

Validation status distribution:


,count
validation_status,
<NA>,125240
approved,115396
created1,7043
processing,678



Trust score distribution:


,count
trust_score,
0.5,125240
0.6,7721
1.0,115396



Top violation labels:


,violation,count
0,WRONG PARKING,134359
1,NO PARKING,118435
2,PARKING IN A MAIN ROAD,19971
3,DEFECTIVE NUMBER PLATE,6121
4,PARKING ON FOOTPATH,3108
5,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,1895
6,DOUBLE PARKING,1623
7,PARKING NEAR ROAD CROSSING,1370
8,REFUSE TO GO FOR HIRE,620
9,PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS,411



Top vehicle types:


,count
vehicle_type_final,
SCOOTER,79189
CAR,75515
MOTOR CYCLE,34212
PASSENGER AUTO,29392
MAXI-CAB,10242
LGV,6777
GOODS AUTO,2425
MOPED,1752
PRIVATE BUS,1282


,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,modified_datetime,...,month,year_month,is_weekend,time_bucket_3h,is_peak_hour,violation_list,offence_code_list,n_violation_types,vehicle_type_final,is_camera_junction
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,2023-11-28 04:48:04.582978+00,...,11,2023-11,0,3,0,"[WRONG PARKING, PARKING NEAR ROAD CROSSING]","[112, 104]",2,MAXI-CAB,0
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,2023-11-24 23:00:24.115257+00,...,11,2023-11,1,3,0,[NO PARKING],[113],1,CAR,0
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,2023-11-28 04:47:02.33776+00,...,11,2023-11,0,3,0,"[WRONG PARKING, PARKING IN A MAIN ROAD]","[112, 107]",2,MAXI-CAB,0
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,2023-11-18 04:46:57.216868+00,...,11,2023-11,0,12,0,[NO PARKING],[113],1,SCOOTER,0
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,2023-11-28 02:44:50.46737+00,...,11,2023-11,0,9,1,[NO PARKING],[113],1,TANKER,1


In [5]:
# =========================
# VEHICLE OBSTRUCTION WEIGHTS
# =========================

VEHICLE_OBSTRUCTION_WEIGHT = {
    "SCOOTER": 1.0,
    "MOTOR CYCLE": 1.0,
    "MOPED": 1.0,

    "PASSENGER AUTO": 1.7,
    "GOODS AUTO": 1.8,

    "CAR": 2.0,
    "JEEP": 2.0,

    "VAN": 2.5,
    "MAXI-CAB": 2.5,
    "TEMPO": 2.8,
    "LGV": 3.0,
    "MINI LORRY": 3.0,

    "PRIVATE BUS": 4.0,
    "TOURIST BUS": 4.0,
    "SCHOOL VEHICLE": 4.0,
    "FACTORY BUS": 4.0,
    "BUS (BMTC/KSRTC)": 4.2,

    "HGV": 4.5,
    "LORRY/GOODS VEHICLE": 4.5,
    "TANKER": 4.5,
    "TRACTOR": 4.0,

    "OTHERS": 2.0,
}

DEFAULT_VEHICLE_WEIGHT = 2.0

# =========================
# VIOLATION SEVERITY WEIGHTS
# =========================

VIOLATION_SEVERITY_WEIGHT = {
    "NO PARKING": 1.0,
    "WRONG PARKING": 1.1,

    "PARKING IN A MAIN ROAD": 2.2,
    "DOUBLE PARKING": 2.5,
    "PARKING NEAR ROAD CROSSING": 2.3,
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS": 2.4,
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE": 2.0,

    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC": 1.8,
    "PARKING OTHER THAN BUS STOP": 1.5,
    "PARKING ON FOOTPATH": 1.4,

    "DEFECTIVE NUMBER PLATE": 0.3,
    "USING BLACK FILM/OTHER MATERIALS": 0.2,
    "WITHOUT SIDE MIRROR": 0.2,
    "REFUSE TO GO FOR HIRE": 0.2,
    "DEMANDING EXCESS FARE": 0.2,
}

DEFAULT_VIOLATION_SEVERITY = 0.8

CAMERA_JUNCTION_MULTIPLIER = 1.25
PEAK_HOUR_MULTIPLIER = 1.20
WEEKEND_MULTIPLIER = 1.05

# =========================
# HELPER FUNCTIONS
# =========================

def has_violation(label, labels):
    return int(label in labels)

def max_violation_weight(labels):
    if not labels:
        return DEFAULT_VIOLATION_SEVERITY

    return max(
        VIOLATION_SEVERITY_WEIGHT.get(label, DEFAULT_VIOLATION_SEVERITY)
        for label in labels
    )

def sum_violation_weight(labels):
    if not labels:
        return DEFAULT_VIOLATION_SEVERITY

    return sum(
        VIOLATION_SEVERITY_WEIGHT.get(label, DEFAULT_VIOLATION_SEVERITY)
        for label in labels
    )

# =========================
# VIOLATION FLAGS
# =========================

df["is_no_parking"] = df["violation_list"].apply(lambda x: has_violation("NO PARKING", x))
df["is_wrong_parking"] = df["violation_list"].apply(lambda x: has_violation("WRONG PARKING", x))

df["is_main_road"] = df["violation_list"].apply(lambda x: has_violation("PARKING IN A MAIN ROAD", x))
df["is_double_parking"] = df["violation_list"].apply(lambda x: has_violation("DOUBLE PARKING", x))
df["is_near_crossing"] = df["violation_list"].apply(lambda x: has_violation("PARKING NEAR ROAD CROSSING", x))
df["is_near_signal_zebra"] = df["violation_list"].apply(
    lambda x: has_violation("PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS", x)
)
df["is_opposite_parked_vehicle"] = df["violation_list"].apply(
    lambda x: has_violation("PARKING OPPOSITE TO ANOTHER PARKED VEHICLE", x)
)

df["is_footpath"] = df["violation_list"].apply(lambda x: has_violation("PARKING ON FOOTPATH", x))
df["is_near_bus_stop_school_hospital"] = df["violation_list"].apply(
    lambda x: has_violation("PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC", x)
)

df["has_high_obstruction_violation"] = (
    (df["is_main_road"] == 1)
    | (df["is_double_parking"] == 1)
    | (df["is_near_crossing"] == 1)
    | (df["is_near_signal_zebra"] == 1)
    | (df["is_opposite_parked_vehicle"] == 1)
).astype(int)

# =========================
# IMPACT SCORE
# =========================

df["vehicle_obstruction_weight"] = (
    df["vehicle_type_final"]
    .map(VEHICLE_OBSTRUCTION_WEIGHT)
    .fillna(DEFAULT_VEHICLE_WEIGHT)
    .astype(float)
)

df["violation_severity_max"] = df["violation_list"].apply(max_violation_weight).astype(float)
df["violation_severity_sum"] = df["violation_list"].apply(sum_violation_weight).astype(float)
df["violation_severity_sum_capped"] = df["violation_severity_sum"].clip(upper=5.0)

df["context_multiplier"] = 1.0
df["context_multiplier"] *= np.where(df["is_camera_junction"] == 1, CAMERA_JUNCTION_MULTIPLIER, 1.0)
df["context_multiplier"] *= np.where(df["is_peak_hour"] == 1, PEAK_HOUR_MULTIPLIER, 1.0)
df["context_multiplier"] *= np.where(df["is_weekend"] == 1, WEEKEND_MULTIPLIER, 1.0)

df["parking_impact_score"] = (
    df["trust_score"]
    * df["vehicle_obstruction_weight"]
    * df["violation_severity_max"]
    * df["context_multiplier"]
)

df["parking_impact_score_multi"] = (
    df["trust_score"]
    * df["vehicle_obstruction_weight"]
    * df["violation_severity_sum_capped"]
    * df["context_multiplier"]
)

p99 = df["parking_impact_score"].quantile(0.99)

if pd.notna(p99) and p99 > 0:
    df["parking_impact_score_norm"] = (df["parking_impact_score"] / p99).clip(upper=1.0)
else:
    df["parking_impact_score_norm"] = 0.0

# =========================
# SANITY CHECKS
# =========================

print("Impact scoring complete.")
print("Shape:", df.shape)

print("\nHigh obstruction violation rate:")
display(df["has_high_obstruction_violation"].value_counts(normalize=True).to_frame("share"))

print("\nVehicle obstruction weight distribution:")
display(df["vehicle_obstruction_weight"].describe())

print("\nParking impact score distribution:")
display(df["parking_impact_score"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nTop rows by parking impact score:")
display(
    df[
        [
            "created_ist_naive",
            "latitude",
            "longitude",
            "police_station",
            "junction_name",
            "vehicle_type_final",
            "violation_list",
            "trust_score",
            "vehicle_obstruction_weight",
            "violation_severity_max",
            "context_multiplier",
            "parking_impact_score",
        ]
    ]
    .sort_values("parking_impact_score", ascending=False)
    .head(10)
)

Impact scoring complete.
Shape: (248357, 54)

High obstruction violation rate:


,share
has_high_obstruction_violation,
0,0.911724
1,0.088276



Vehicle obstruction weight distribution:


count    248357.000000
mean          1.597475
std           0.674516
min           1.000000
25%           1.000000
50%           1.700000
75%           2.000000
max           4.500000
Name: vehicle_obstruction_weight, dtype: float64


Parking impact score distribution:


count    248357.000000
mean          1.664125
std           1.160783
min           0.500000
50%           1.320000
75%           2.200000
90%           2.887500
95%           3.609375
99%           5.775000
max          17.718750
Name: parking_impact_score, dtype: float64


Top rows by parking impact score:


,created_ist_naive,latitude,longitude,police_station,junction_name,vehicle_type_final,violation_list,trust_score,vehicle_obstruction_weight,violation_severity_max,context_multiplier,parking_impact_score
184522,2023-12-31 09:17:46,12.926585,77.586254,Jayanagara,"BTP226 - 33rd Cross, 11th Main, Jayanagar",HGV,"[WRONG PARKING, NO PARKING, DOUBLE PARKING, PA...",1.0,4.5,2.5,1.575,17.71875
138399,2023-11-15 08:09:46,12.960444,77.577594,City Market,BTP104 - Medical College Circle,PRIVATE BUS,"[NO PARKING, WRONG PARKING, DOUBLE PARKING, PA...",1.0,4.0,2.5,1.500,15.00000
188557,2024-01-05 08:57:46,12.968823,77.602008,Ashok Nagar,"BTP099 - Cash Pharmacy, Residency Road",SCHOOL VEHICLE,"[DOUBLE PARKING, PARKING IN A MAIN ROAD, NO PA...",1.0,4.0,2.5,1.500,15.00000
138382,2023-11-15 08:00:46,12.959672,77.577681,City Market,BTP104 - Medical College Circle,PRIVATE BUS,"[WRONG PARKING, NO PARKING, DOUBLE PARKING]",1.0,4.0,2.5,1.500,15.00000
15727,2024-01-12 10:45:46,12.928435,77.597383,Jayanagara,"BTP127 - Sagar Hospital, Jayanagar",SCHOOL VEHICLE,"[WRONG PARKING, NO PARKING, DOUBLE PARKING, PA...",1.0,4.0,2.5,1.500,15.00000
137590,2023-11-15 10:33:46,12.959857,77.577709,City Market,BTP104 - Medical College Circle,LORRY/GOODS VEHICLE,"[WRONG PARKING, PARKING IN A MAIN ROAD]",1.0,4.5,2.2,1.500,14.85000
245673,2023-11-29 10:33:46,12.970232,77.574907,City Market,BTP043 - Upparpet Junction,LORRY/GOODS VEHICLE,"[PARKING IN A MAIN ROAD, WRONG PARKING]",1.0,4.5,2.2,1.500,14.85000
140394,2023-11-14 08:52:46,12.958039,77.582319,City Market,BTP167 - JC Road Junction,HGV,"[PARKING IN A MAIN ROAD, WRONG PARKING]",1.0,4.5,2.2,1.500,14.85000
126134,2023-12-04 08:30:46,12.943199,77.530807,Byatarayanapura,BTP098 - Nayandahalli Junction,HGV,"[NO PARKING, PARKING IN A MAIN ROAD]",1.0,4.5,2.2,1.500,14.85000
124261,2023-12-04 08:29:46,12.943199,77.530807,Byatarayanapura,BTP098 - Nayandahalli Junction,HGV,"[PARKING IN A MAIN ROAD, NO PARKING, DEFECTIVE...",1.0,4.5,2.2,1.500,14.85000


In [6]:
# =========================
# H3 SETUP WITH GRID FALLBACK
# =========================

H3_AVAILABLE = False

try:
    import h3

    H3_AVAILABLE = True
    print("H3 available. Using H3 fallback cells where applicable.")
except ImportError:
    print("H3 not installed. Using rounded lat/lon grid fallback cells instead.")

def latlon_to_h3(lat, lon, resolution):
    if pd.isna(lat) or pd.isna(lon) or not H3_AVAILABLE:
        return np.nan

    try:
        return h3.latlng_to_cell(lat, lon, resolution)  # h3-py v4
    except AttributeError:
        return h3.geo_to_h3(lat, lon, resolution)       # older h3-py

def latlon_to_grid(lat, lon, grid_size):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan

    lat_bin = round(lat / grid_size) * grid_size
    lon_bin = round(lon / grid_size) * grid_size

    return f"grid_{lat_bin:.5f}_{lon_bin:.5f}"

# =========================
# BASE SPATIAL COLUMNS
# =========================

junction_clean = (
    df["junction_name"]
    .astype(str)
    .str.strip()
)

df["has_named_junction"] = (
    junction_clean.notna()
    & junction_clean.ne("")
    & junction_clean.str.lower().ne("no junction")
    & junction_clean.str.lower().ne("nan")
)

df["junction_unit_id"] = np.where(
    df["has_named_junction"],
    "junction_" + junction_clean,
    np.nan,
)

df["h3_cell"] = [
    latlon_to_h3(lat, lon, H3_RESOLUTION)
    for lat, lon in zip(df["latitude"], df["longitude"])
]

df["grid_cell"] = [
    latlon_to_grid(lat, lon, GRID_SIZE_DEGREES)
    for lat, lon in zip(df["latitude"], df["longitude"])
]

df["h3_or_grid_cell"] = df["h3_cell"].fillna(df["grid_cell"])

# =========================
# FINAL SPATIAL UNIT
# =========================

if SPATIAL_MODE == "junction_plus_h3":
    df["spatial_unit_id"] = np.where(
        df["has_named_junction"],
        df["junction_unit_id"],
        "h3_" + df["h3_or_grid_cell"].astype(str),
    )

elif SPATIAL_MODE == "h3_only":
    df["spatial_unit_id"] = "h3_" + df["h3_or_grid_cell"].astype(str)

elif SPATIAL_MODE == "grid_only":
    df["spatial_unit_id"] = df["grid_cell"]

elif SPATIAL_MODE == "junction_only":
    df["spatial_unit_id"] = np.where(
        df["has_named_junction"],
        df["junction_unit_id"],
        df["grid_cell"],
    )

else:
    raise ValueError(
        "Invalid SPATIAL_MODE. Choose one of: "
        "'junction_plus_h3', 'h3_only', 'grid_only', 'junction_only'"
    )

df = df[df["spatial_unit_id"].notna()].copy()

# =========================
# UNIT CENTROIDS AND SUMMARY
# =========================

spatial_centroids = (
    df.groupby("spatial_unit_id", as_index=False)
    .agg(
        unit_latitude=("latitude", "median"),
        unit_longitude=("longitude", "median"),
        unit_event_count=("latitude", "size"),
        unit_impact_sum=("parking_impact_score", "sum"),
        unit_avg_impact=("parking_impact_score", "mean"),
        named_junction_share=("has_named_junction", "mean"),
    )
)

df = df.merge(
    spatial_centroids[
        [
            "spatial_unit_id",
            "unit_latitude",
            "unit_longitude",
        ]
    ],
    on="spatial_unit_id",
    how="left",
)

# =========================
# SANITY CHECKS
# =========================

print("Spatial unit creation complete.")
print("Spatial mode:", SPATIAL_MODE)
print("H3 available:", H3_AVAILABLE)
print("Rows:", len(df))
print("Spatial units:", df["spatial_unit_id"].nunique())
print("Named junction rows:", int(df["has_named_junction"].sum()))
print("Non-junction rows:", int((~df["has_named_junction"]).sum()))

unit_counts = df["spatial_unit_id"].value_counts()

print("\nSpatial unit size distribution:")
display(unit_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nTop spatial units by event count:")
display(
    spatial_centroids
    .sort_values("unit_event_count", ascending=False)
    .head(15)
)

print("\nTop spatial units by total impact score:")
display(
    spatial_centroids
    .sort_values("unit_impact_sum", ascending=False)
    .head(15)
)

H3 available. Using H3 fallback cells where applicable.
Spatial unit creation complete.
Spatial mode: junction_plus_h3
H3 available: True
Rows: 248357
Spatial units: 2156
Named junction rows: 125018
Non-junction rows: 123339

Spatial unit size distribution:


count     2156.000000
mean       115.193414
std        544.661132
min          1.000000
50%         10.000000
75%         47.000000
90%        201.500000
95%        473.250000
99%       1901.150000
max      12006.000000
Name: count, dtype: float64


Top spatial units by event count:


,spatial_unit_id,unit_latitude,unit_longitude,unit_event_count,unit_impact_sum,unit_avg_impact,named_junction_share
2022,junction_BTP051 - Safina Plaza Junction,12.981220,77.608711,12006,20493.297200,1.706921,1.0
2044,junction_BTP082 - KR Market Junction,12.964379,77.576965,9671,14904.853813,1.541191,1.0
2014,junction_BTP040 - Elite Junction,12.976935,77.576479,9385,14339.821275,1.527951,1.0
2018,junction_BTP044 - Sagar Theatre Junction,12.974472,77.578938,9056,15694.814125,1.733085,1.0
797,h3_89618920923ffff,12.933571,77.691013,5627,6515.969950,1.157983,0.0
2027,junction_BTP058 - Subbanna Junction,12.978968,77.578605,4519,8905.841650,1.970755,1.0
2142,junction_BTP211 - Central Street Junction,12.984038,77.603449,4170,8093.535063,1.940896,1.0
2004,junction_BTP027 - Modi Bridge Junction,12.999177,77.549582,3790,5056.189750,1.334087,1.0
2026,junction_BTP057 - Anand Rao Junction,12.979291,77.574870,3312,7239.780013,2.185924,1.0
1998,junction_BTP020 - Hosahalli Metro Station,12.974416,77.545242,3136,3827.773875,1.220591,1.0



Top spatial units by total impact score:


,spatial_unit_id,unit_latitude,unit_longitude,unit_event_count,unit_impact_sum,unit_avg_impact,named_junction_share
2022,junction_BTP051 - Safina Plaza Junction,12.981220,77.608711,12006,20493.297200,1.706921,1.0
2018,junction_BTP044 - Sagar Theatre Junction,12.974472,77.578938,9056,15694.814125,1.733085,1.0
2044,junction_BTP082 - KR Market Junction,12.964379,77.576965,9671,14904.853813,1.541191,1.0
2014,junction_BTP040 - Elite Junction,12.976935,77.576479,9385,14339.821275,1.527951,1.0
2027,junction_BTP058 - Subbanna Junction,12.978968,77.578605,4519,8905.841650,1.970755,1.0
2142,junction_BTP211 - Central Street Junction,12.984038,77.603449,4170,8093.535063,1.940896,1.0
2026,junction_BTP057 - Anand Rao Junction,12.979291,77.574870,3312,7239.780013,2.185924,1.0
797,h3_89618920923ffff,12.933571,77.691013,5627,6515.969950,1.157983,0.0
2045,"junction_BTP083 - AS Char Street, Mysore Road",12.966414,77.574508,2319,5501.746938,2.372465,1.0
2019,junction_BTP045 - Danvanthri Road Junction,12.977611,77.574786,2605,5249.054063,2.014992,1.0


In [7]:
# =========================
# DAILY PANEL CONFIG
# =========================

SMOOTHING_ALPHA = 3.0
USE_APPROVED_WEIGHTING = True

# =========================
# HELPERS
# =========================

def mode_or_nan(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

# =========================
# DATE RANGE
# =========================

df["date"] = pd.to_datetime(df["date"])

date_min = df["date"].min()
date_max = df["date"].max()
all_dates = pd.date_range(date_min, date_max, freq="D")

print("Analysis date range:", date_min.date(), "to", date_max.date())
print("Total analysis days:", len(all_dates))

# =========================
# OBSERVED DAILY AGGREGATION
# =========================

daily_obs = (
    df.groupby(["spatial_unit_id", "date"], as_index=False)
    .agg(
        raw_violation_count=("latitude", "size"),
        trust_weighted_count=("trust_score", "sum"),

        impact_sum=("parking_impact_score", "sum"),
        impact_multi_sum=("parking_impact_score_multi", "sum"),
        avg_impact=("parking_impact_score", "mean"),

        high_obstruction_count=("has_high_obstruction_violation", "sum"),
        main_road_count=("is_main_road", "sum"),
        double_parking_count=("is_double_parking", "sum"),
        near_crossing_count=("is_near_crossing", "sum"),
        near_signal_zebra_count=("is_near_signal_zebra", "sum"),
        footpath_count=("is_footpath", "sum"),

        peak_hour_count=("is_peak_hour", "sum"),
        weekend_event_count=("is_weekend", "sum"),

        unique_vehicles=("vehicle_number", "nunique"),
        unique_devices=("device_id", "nunique"),

        unit_latitude=("unit_latitude", "median"),
        unit_longitude=("unit_longitude", "median"),
        police_station_mode=("police_station", mode_or_nan),
    )
)

# =========================
# STATIC SPATIAL UNIT TABLE
# =========================

spatial_units = (
    df.groupby("spatial_unit_id", as_index=False)
    .agg(
        unit_latitude=("unit_latitude", "median"),
        unit_longitude=("unit_longitude", "median"),
        total_events=("latitude", "size"),
        total_impact=("parking_impact_score", "sum"),
        named_junction_share=("has_named_junction", "mean"),
        police_station_mode=("police_station", mode_or_nan),
    )
)

# =========================
# FULL DAILY PANEL INCLUDING ZERO DAYS
# =========================

panel_index = (
    spatial_units[["spatial_unit_id"]]
    .assign(key=1)
    .merge(pd.DataFrame({"date": all_dates, "key": 1}), on="key")
    .drop(columns="key")
)

daily_panel = panel_index.merge(
    daily_obs,
    on=["spatial_unit_id", "date"],
    how="left",
)

daily_panel = daily_panel.merge(
    spatial_units,
    on="spatial_unit_id",
    how="left",
    suffixes=("", "_unit"),
)

for col in ["unit_latitude", "unit_longitude", "police_station_mode"]:
    daily_panel[col] = daily_panel[col].fillna(daily_panel[f"{col}_unit"])
    daily_panel = daily_panel.drop(columns=[f"{col}_unit"])

event_cols = [
    "raw_violation_count",
    "trust_weighted_count",
    "impact_sum",
    "impact_multi_sum",
    "avg_impact",
    "high_obstruction_count",
    "main_road_count",
    "double_parking_count",
    "near_crossing_count",
    "near_signal_zebra_count",
    "footpath_count",
    "peak_hour_count",
    "weekend_event_count",
    "unique_vehicles",
    "unique_devices",
]

daily_panel[event_cols] = daily_panel[event_cols].fillna(0)

# =========================
# CALENDAR FEATURES
# =========================

daily_panel["dayofweek"] = daily_panel["date"].dt.dayofweek
daily_panel["month"] = daily_panel["date"].dt.month
daily_panel["year_month"] = daily_panel["date"].dt.to_period("M").astype(str)
daily_panel["is_weekend_day"] = daily_panel["dayofweek"].isin([5, 6]).astype(int)

if USE_APPROVED_WEIGHTING:
    daily_panel["daily_violation_signal"] = daily_panel["trust_weighted_count"]
    daily_panel["daily_impact_signal"] = daily_panel["impact_sum"]
else:
    daily_panel["daily_violation_signal"] = daily_panel["raw_violation_count"]
    daily_panel["daily_impact_signal"] = daily_panel["impact_multi_sum"]

# =========================
# FULL-PERIOD UNIT SUMMARY
# Used only as static descriptive context.
# Model confidence is recalculated history-only later.
# =========================

unit_summary = (
    daily_panel.groupby("spatial_unit_id", as_index=False)
    .agg(
        unit_latitude=("unit_latitude", "median"),
        unit_longitude=("unit_longitude", "median"),
        police_station_mode=("police_station_mode", "first"),

        total_events=("raw_violation_count", "sum"),
        total_trust_weighted_events=("trust_weighted_count", "sum"),
        total_impact=("impact_sum", "sum"),

        active_days=("raw_violation_count", lambda x: (x > 0).sum()),
        observed_days=("date", "nunique"),

        mean_daily_count=("raw_violation_count", "mean"),
        mean_daily_trust_count=("trust_weighted_count", "mean"),
        mean_daily_impact=("impact_sum", "mean"),

        max_daily_count=("raw_violation_count", "max"),
        max_daily_impact=("impact_sum", "max"),

        high_obstruction_total=("high_obstruction_count", "sum"),
        named_junction_share=("named_junction_share", "first"),
    )
)

city_mean_daily_impact = unit_summary["mean_daily_impact"].mean()
city_mean_daily_count = unit_summary["mean_daily_count"].mean()

unit_summary["smoothed_daily_impact"] = (
    (unit_summary["mean_daily_impact"] * unit_summary["observed_days"])
    + (city_mean_daily_impact * SMOOTHING_ALPHA)
) / (unit_summary["observed_days"] + SMOOTHING_ALPHA)

unit_summary["smoothed_daily_count"] = (
    (unit_summary["mean_daily_count"] * unit_summary["observed_days"])
    + (city_mean_daily_count * SMOOTHING_ALPHA)
) / (unit_summary["observed_days"] + SMOOTHING_ALPHA)

unit_summary["active_day_rate"] = unit_summary["active_days"] / unit_summary["observed_days"]

unit_summary["confidence_score_full_period"] = (
    np.sqrt(unit_summary["total_events"].clip(lower=0))
    / np.sqrt(unit_summary["total_events"].max())
)

unit_summary["confidence_score_full_period"] = (
    0.7 * unit_summary["confidence_score_full_period"]
    + 0.3 * unit_summary["active_day_rate"]
).clip(0, 1)

# =========================
# SANITY CHECKS
# =========================

print("Daily panel complete.")
print("daily_obs shape:", daily_obs.shape)
print("daily_panel shape:", daily_panel.shape)
print("unit_summary shape:", unit_summary.shape)

print("\nDaily signal distribution:")
display(
    daily_panel[
        [
            "raw_violation_count",
            "trust_weighted_count",
            "impact_sum",
            "daily_violation_signal",
            "daily_impact_signal",
        ]
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

print("\nTop units by smoothed daily impact:")
display(
    unit_summary
    .sort_values("smoothed_daily_impact", ascending=False)
    .head(15)[
        [
            "spatial_unit_id",
            "police_station_mode",
            "unit_latitude",
            "unit_longitude",
            "total_events",
            "active_days",
            "mean_daily_impact",
            "smoothed_daily_impact",
            "confidence_score_full_period",
        ]
    ]
)

Analysis date range: 2023-11-10 to 2024-04-08
Total analysis days: 151
Daily panel complete.
daily_obs shape: (36005, 20)
daily_panel shape: (325556, 29)
unit_summary shape: (2156, 20)

Daily signal distribution:


,raw_violation_count,trust_weighted_count,impact_sum,daily_violation_signal,daily_impact_signal
count,325556.000000,325556.000000,325556.000000,325556.000000,325556.000000
mean,0.762870,0.561036,1.269511,0.561036,1.269511
std,5.148616,3.886905,8.882714,3.886905,8.882714
min,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000
90%,1.000000,0.500000,1.100000,0.500000,1.100000
95%,3.000000,2.000000,4.950000,2.000000,4.950000
99%,17.000000,13.000000,28.500000,13.000000,28.500000
max,323.000000,300.500000,689.317125,300.500000,689.317125



Top units by smoothed daily impact:


,spatial_unit_id,police_station_mode,unit_latitude,unit_longitude,total_events,active_days,mean_daily_impact,smoothed_daily_impact,confidence_score_full_period
2022,junction_BTP051 - Safina Plaza Junction,Shivajinagar,12.981220,77.608711,12006.0,151,135.717200,133.098089,1.000000
2018,junction_BTP044 - Sagar Theatre Junction,Upparpet,12.974472,77.578938,9056.0,151,103.939166,101.939108,0.907949
2044,junction_BTP082 - KR Market Junction,City Market,12.964379,77.576965,9671.0,151,98.707641,96.809496,0.928253
2014,junction_BTP040 - Elite Junction,Upparpet,12.976935,77.576479,9385.0,151,94.965704,93.140453,0.918894
2027,junction_BTP058 - Subbanna Junction,Upparpet,12.978968,77.578605,4519.0,149,58.979084,57.854871,0.725484
2142,junction_BTP211 - Central Street Junction,Shivajinagar,12.984038,77.603449,4170.0,149,53.599570,52.580153,0.708567
2026,junction_BTP057 - Anand Rao Junction,Upparpet,12.979291,77.574870,3312.0,147,47.945563,47.036289,0.659711
797,h3_89618920923ffff,HAL Old Airport,12.933571,77.691013,5627.0,95,43.152119,42.336224,0.667964
2045,"junction_BTP083 - AS Char Street, Mysore Road",Chamarajpet,12.966414,77.574508,2319.0,138,36.435410,35.750360,0.581817
2019,junction_BTP045 - Danvanthri Road Junction,Upparpet,12.977611,77.574786,2605.0,147,34.761947,34.109497,0.618117


In [8]:
# =========================
# SORT PANEL
# =========================

daily_panel = daily_panel.sort_values(["spatial_unit_id", "date"]).reset_index(drop=True)

# =========================
# LAG FEATURES
# =========================

for lag in LAG_DAYS:
    daily_panel[f"{TARGET_COL}_lag_{lag}d"] = (
        daily_panel.groupby("spatial_unit_id")[TARGET_COL].shift(lag)
    )

    daily_panel[f"count_lag_{lag}d"] = (
        daily_panel.groupby("spatial_unit_id")["raw_violation_count"].shift(lag)
    )

# =========================
# ROLLING FEATURES
# Shift first, then roll, so same-day data is not used.
# =========================

for window in ROLL_WINDOWS:
    shifted_target = daily_panel.groupby("spatial_unit_id")[TARGET_COL].shift(1)
    shifted_count = daily_panel.groupby("spatial_unit_id")["raw_violation_count"].shift(1)
    shifted_high_obstruction = daily_panel.groupby("spatial_unit_id")["high_obstruction_count"].shift(1)
    shifted_peak = daily_panel.groupby("spatial_unit_id")["peak_hour_count"].shift(1)

    daily_panel[f"{TARGET_COL}_roll_sum_{window}d"] = (
        shifted_target
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    daily_panel[f"{TARGET_COL}_roll_mean_{window}d"] = (
        shifted_target
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    daily_panel[f"count_roll_sum_{window}d"] = (
        shifted_count
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    daily_panel[f"count_roll_active_days_{window}d"] = (
        (shifted_count > 0)
        .astype(float)
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    daily_panel[f"high_obstruction_roll_sum_{window}d"] = (
        shifted_high_obstruction
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    daily_panel[f"peak_hour_roll_sum_{window}d"] = (
        shifted_peak
        .groupby(daily_panel["spatial_unit_id"])
        .rolling(window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

# =========================
# MOMENTUM / MIX FEATURES
# =========================

daily_panel["impact_momentum_7_vs_30"] = (
    daily_panel[f"{TARGET_COL}_roll_mean_7d"]
    - daily_panel[f"{TARGET_COL}_roll_mean_30d"]
)

daily_panel["count_momentum_7_vs_30"] = (
    daily_panel["count_roll_sum_7d"] / 7
    - daily_panel["count_roll_sum_30d"] / 30
)

daily_panel["recent_persistence_7d"] = daily_panel["count_roll_active_days_7d"] / 7
daily_panel["recent_persistence_30d"] = daily_panel["count_roll_active_days_30d"] / 30

daily_panel["high_obstruction_share_30d"] = (
    daily_panel["high_obstruction_roll_sum_30d"]
    / daily_panel["count_roll_sum_30d"].replace(0, np.nan)
).fillna(0)

daily_panel["peak_hour_share_30d"] = (
    daily_panel["peak_hour_roll_sum_30d"]
    / daily_panel["count_roll_sum_30d"].replace(0, np.nan)
).fillna(0)

# =========================
# CORRECT FORWARD TARGET
# target at date t = sum of target signal from t+1 to t+7
# =========================

def forward_rolling_sum(s, horizon):
    out = pd.Series(0.0, index=s.index)

    for k in range(1, horizon + 1):
        out = out + s.shift(-k).fillna(0)

    return out

target_name = f"future_{FORECAST_HORIZON_DAYS}d_impact"
log_target_name = f"log_future_{FORECAST_HORIZON_DAYS}d_impact"

daily_panel[target_name] = (
    daily_panel
    .groupby("spatial_unit_id", group_keys=False)[TARGET_COL]
    .apply(lambda s: forward_rolling_sum(s, FORECAST_HORIZON_DAYS))
)

daily_panel[log_target_name] = np.log1p(daily_panel[target_name])

max_trainable_date = daily_panel["date"].max() - pd.Timedelta(days=FORECAST_HORIZON_DAYS)

daily_panel["has_full_future_horizon"] = (
    daily_panel["date"] <= max_trainable_date
).astype(int)

# =========================
# ADD STATIC UNIT FEATURES
# =========================

unit_static_cols = [
    "spatial_unit_id",
    "total_events",
    "total_impact",
    "active_days",
    "active_day_rate",
    "mean_daily_count",
    "mean_daily_impact",
    "smoothed_daily_count",
    "smoothed_daily_impact",
    "confidence_score_full_period",
    "named_junction_share",
]

daily_model = daily_panel.merge(
    unit_summary[unit_static_cols],
    on="spatial_unit_id",
    how="left",
    suffixes=("", "_unit"),
)

# =========================
# FILL FEATURE NANS
# =========================

feature_fill_cols = [
    c for c in daily_model.columns
    if any(token in c for token in ["lag_", "roll_", "momentum", "persistence", "share"])
]

daily_model[feature_fill_cols] = daily_model[feature_fill_cols].fillna(0)

# =========================
# SANITY CHECKS
# =========================

print("Lag, rolling, trend, and target features complete.")
print("daily_model shape:", daily_model.shape)
print("Target:", target_name)
print("Max trainable date:", max_trainable_date.date())

print("\nFuture target distribution:")
display(
    daily_model.loc[
        daily_model["has_full_future_horizon"] == 1,
        [target_name, log_target_name],
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

print("\nManual target sanity check for one unit with future activity:")

sample_unit = (
    daily_model.loc[daily_model[target_name] > 0]
    .groupby("spatial_unit_id")[target_name]
    .max()
    .sort_values(ascending=False)
    .index[0]
)

sample = daily_model[daily_model["spatial_unit_id"] == sample_unit].copy()

first_signal_date = sample.loc[sample[TARGET_COL] > 0, "date"].min()
window_start = first_signal_date - pd.Timedelta(days=10)
window_end = first_signal_date + pd.Timedelta(days=5)

display(
    sample[
        (sample["date"] >= window_start)
        & (sample["date"] <= window_end)
    ][
        [
            "spatial_unit_id",
            "date",
            TARGET_COL,
            f"{TARGET_COL}_lag_1d",
            f"{TARGET_COL}_roll_sum_7d",
            target_name,
        ]
    ]
)

Lag, rolling, trend, and target features complete.
daily_model shape: (325556, 72)
Target: future_7d_impact
Max trainable date: 2024-04-01

Future target distribution:


,future_7d_impact,log_future_7d_impact
count,310464.000000,310464.000000
mean,8.887926,0.750229
std,47.468043,1.289902
min,0.000000,0.000000
50%,0.000000,0.000000
75%,2.310000,1.196948
90%,15.480000,2.802148
95%,37.839400,3.659435
99%,156.985587,5.062504
max,2440.058437,7.800187



Manual target sanity check for one unit with future activity:


,spatial_unit_id,date,daily_impact_signal,daily_impact_signal_lag_1d,daily_impact_signal_roll_sum_7d,future_7d_impact
305322,junction_BTP051 - Safina Plaza Junction,2023-11-10,95.442500,0.000000,0.000000,1867.785563
305323,junction_BTP051 - Safina Plaza Junction,2023-11-11,281.192625,95.442500,95.442500,2115.147188
305324,junction_BTP051 - Safina Plaza Junction,2023-11-12,424.392938,281.192625,376.635125,2284.485938
305325,junction_BTP051 - Safina Plaza Junction,2023-11-13,134.210000,424.392938,801.028063,2403.199688
305326,junction_BTP051 - Safina Plaza Junction,2023-11-14,126.200000,134.210000,935.238063,2371.842188
305327,junction_BTP051 - Safina Plaza Junction,2023-11-15,175.905000,126.200000,1061.438063,2337.605937


In [9]:
from scipy.stats import spearmanr

TARGET = f"future_{FORECAST_HORIZON_DAYS}d_impact"
LOG_TARGET = f"log_future_{FORECAST_HORIZON_DAYS}d_impact"

CONFIDENCE_SCALE_EVENTS = 50

# =========================
# HISTORY-ONLY FEATURES
# These are safe because they exclude the current day.
# =========================

daily_model = daily_model.sort_values(["spatial_unit_id", "date"]).reset_index(drop=True)

g = daily_model.groupby("spatial_unit_id", group_keys=False)

daily_model["hist_total_events"] = (
    g["raw_violation_count"].cumsum() - daily_model["raw_violation_count"]
)

daily_model["hist_total_impact"] = (
    g["daily_impact_signal"].cumsum() - daily_model["daily_impact_signal"]
)

daily_model["hist_active_days"] = (
    g["raw_violation_count"].transform(lambda s: (s > 0).cumsum())
    - (daily_model["raw_violation_count"] > 0).astype(int)
)

daily_model["hist_observed_days"] = g.cumcount()

daily_model["hist_active_day_rate"] = (
    daily_model["hist_active_days"]
    / daily_model["hist_observed_days"].replace(0, np.nan)
).fillna(0)

daily_model["hist_mean_daily_impact"] = (
    daily_model["hist_total_impact"]
    / daily_model["hist_observed_days"].replace(0, np.nan)
).fillna(0)

daily_model["hist_confidence_score"] = (
    np.sqrt(daily_model["hist_total_events"])
    / (
        np.sqrt(daily_model["hist_total_events"])
        + np.sqrt(CONFIDENCE_SCALE_EVENTS)
    )
).fillna(0)

daily_model["hist_confidence_score"] = (
    0.75 * daily_model["hist_confidence_score"]
    + 0.25 * daily_model["hist_active_day_rate"]
).clip(0, 1)

daily_model["hist_sufficient_history_flag"] = (
    daily_model["hist_total_events"] >= MIN_HIST_EVENTS_FOR_EVAL
).astype(int)

# =========================
# BASELINE SCORE FUNCTIONS
# =========================

def add_safe_baseline_scores(frame):
    out = frame.copy()

    out["score_recent_7d_safe"] = out[f"{TARGET_COL}_roll_sum_7d"]
    out["score_recent_30d_safe"] = out[f"{TARGET_COL}_roll_sum_30d"]

    out["score_blended_recent_safe"] = (
        0.65 * out[f"{TARGET_COL}_roll_sum_7d"]
        + 0.35 * out[f"{TARGET_COL}_roll_sum_30d"]
    )

    out["score_priority_proxy_safe"] = (
        out["score_blended_recent_safe"]
        * (1 + 0.50 * out["recent_persistence_30d"])
        * (1 + 0.50 * out["high_obstruction_share_30d"])
        * (0.50 + 0.50 * out["hist_confidence_score"])
    )

    out["score_emerging_safe"] = (
        out[f"{TARGET_COL}_roll_sum_7d"]
        * (1 + np.maximum(out["impact_momentum_7_vs_30"], 0))
        * (0.50 + 0.50 * out["hist_confidence_score"])
    )

    return out

# =========================
# RANKING METRICS
# =========================

def recall_at_k(y_true, y_score, k):
    tmp = pd.DataFrame({"y_true": y_true, "y_score": y_score})
    actual_top_idx = set(tmp.sort_values("y_true", ascending=False).head(k).index)
    pred_top_idx = set(tmp.sort_values("y_score", ascending=False).head(k).index)

    if not actual_top_idx:
        return np.nan

    return len(actual_top_idx & pred_top_idx) / len(actual_top_idx)

def ndcg_at_k(y_true, y_score, k):
    tmp = pd.DataFrame({"y_true": y_true, "y_score": y_score}).sort_values(
        "y_score",
        ascending=False,
    )

    gains = tmp["y_true"].head(k).to_numpy()
    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains * discounts)

    ideal_gains = np.sort(y_true)[::-1][:k]
    ideal_discounts = 1 / np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = np.sum(ideal_gains * ideal_discounts)

    return dcg / idcg if idcg > 0 else np.nan

def safe_spearman(y_true, y_score):
    if pd.Series(y_true).nunique() <= 1 or pd.Series(y_score).nunique() <= 1:
        return np.nan

    return spearmanr(y_true, y_score).correlation

# =========================
# EVALUATE LEAKAGE-SAFE BASELINES
# =========================

safe_score_cols = [
    "score_recent_7d_safe",
    "score_recent_30d_safe",
    "score_blended_recent_safe",
    "score_priority_proxy_safe",
    "score_emerging_safe",
]

safe_eval_rows = []
safe_snapshots = []

for val_date in VALIDATION_DATES:
    val_date = pd.to_datetime(val_date)

    snap = daily_model[
        (daily_model["date"] == val_date)
        & (daily_model["has_full_future_horizon"] == 1)
        & (daily_model["hist_total_events"] >= MIN_HIST_EVENTS_FOR_EVAL)
    ].copy()

    if snap.empty:
        print(f"Skipping {val_date.date()}: no eligible rows.")
        continue

    snap = add_safe_baseline_scores(snap)
    safe_snapshots.append(snap.assign(validation_date=val_date))

    for score in safe_score_cols:
        row = {
            "validation_date": val_date.date(),
            "score": score,
            "n_units": len(snap),
            "spearman": safe_spearman(snap[TARGET], snap[score]),
        }

        for k in TOP_K_LIST:
            row[f"recall_at_{k}"] = recall_at_k(snap[TARGET], snap[score], k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(snap[TARGET].to_numpy(), snap[score].to_numpy(), k)

        safe_eval_rows.append(row)

safe_eval_results = pd.DataFrame(safe_eval_rows)

if safe_snapshots:
    safe_eval_snapshots = pd.concat(safe_snapshots, ignore_index=True)
else:
    raise ValueError("No baseline validation snapshots were created. Check VALIDATION_DATES.")

safe_baseline_summary = (
    safe_eval_results
    .groupby("score", as_index=False)
    .mean(numeric_only=True)
    .sort_values(["recall_at_50", "ndcg_at_50", "spearman"], ascending=False)
)

BEST_SAFE_BASELINE_SCORE = safe_baseline_summary.iloc[0]["score"]

# =========================
# DISPLAY RESULTS
# =========================

print("Leakage-safe baseline evaluation complete.")
print("Best baseline:", BEST_SAFE_BASELINE_SCORE)

print("\nAverage baseline performance:")
display(safe_baseline_summary)

print("\nPer-date baseline performance:")
display(safe_eval_results.sort_values(["validation_date", "recall_at_50", "ndcg_at_50"], ascending=False))

latest_eval_date = pd.to_datetime(VALIDATION_DATES[-1])

latest_safe_snapshot = safe_eval_snapshots[
    safe_eval_snapshots["validation_date"] == latest_eval_date
].copy()

latest_safe_snapshot = latest_safe_snapshot.sort_values(BEST_SAFE_BASELINE_SCORE, ascending=False)

print(f"\nTop baseline recommendations as of {latest_eval_date.date()}:")
display(
    latest_safe_snapshot.head(25)[
        [
            "spatial_unit_id",
            "police_station_mode",
            "unit_latitude",
            "unit_longitude",
            BEST_SAFE_BASELINE_SCORE,
            TARGET,
            f"{TARGET_COL}_roll_sum_7d",
            f"{TARGET_COL}_roll_sum_30d",
            "recent_persistence_30d",
            "high_obstruction_share_30d",
            "hist_total_events",
            "hist_confidence_score",
        ]
    ]
)

Leakage-safe baseline evaluation complete.
Best baseline: score_priority_proxy_safe

Average baseline performance:


,score,n_units,spearman,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
2,score_priority_proxy_safe,1163.0,0.635327,0.750,0.916456,0.71,0.923926,0.685,0.910458,0.6450,0.898035
0,score_blended_recent_safe,1163.0,0.629898,0.775,0.923010,0.70,0.923629,0.670,0.907795,0.6525,0.898369
3,score_recent_30d_safe,1163.0,0.630384,0.725,0.904902,0.70,0.916457,0.660,0.905064,0.6450,0.892037
4,score_recent_7d_safe,1163.0,0.577097,0.700,0.878212,0.63,0.858000,0.615,0.863924,0.6050,0.855818
1,score_emerging_safe,1163.0,0.563089,0.275,0.484370,0.43,0.568837,0.475,0.627601,0.5100,0.665761



Per-date baseline performance:


,validation_date,score,n_units,spearman,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
17,2024-04-01,score_blended_recent_safe,1368,0.627726,0.8,0.932815,0.72,0.927475,0.70,0.923576,0.67,0.906786
18,2024-04-01,score_priority_proxy_safe,1368,0.633187,0.7,0.903976,0.72,0.925690,0.70,0.923181,0.66,0.903982
15,2024-04-01,score_recent_7d_safe,1368,0.606309,0.7,0.887407,0.56,0.836704,0.66,0.882024,0.65,0.873173
16,2024-04-01,score_recent_30d_safe,1368,0.615271,0.7,0.915289,0.68,0.906860,0.64,0.900675,0.64,0.895539
19,2024-04-01,score_emerging_safe,1368,0.601197,0.3,0.536647,0.44,0.634040,0.54,0.691744,0.59,0.744812
11,2024-03-01,score_recent_30d_safe,1252,0.615533,0.7,0.904407,0.68,0.912502,0.68,0.910712,0.62,0.896679
13,2024-03-01,score_priority_proxy_safe,1252,0.616263,0.7,0.939186,0.72,0.926854,0.68,0.906270,0.63,0.909042
12,2024-03-01,score_blended_recent_safe,1252,0.613536,0.7,0.932518,0.72,0.924771,0.66,0.899944,0.62,0.900846
10,2024-03-01,score_recent_7d_safe,1252,0.561406,0.7,0.929679,0.60,0.856418,0.58,0.863458,0.58,0.857600
14,2024-03-01,score_emerging_safe,1252,0.552492,0.4,0.691540,0.52,0.722559,0.36,0.682236,0.46,0.712063



Top baseline recommendations as of 2024-04-01:


,spatial_unit_id,police_station_mode,unit_latitude,unit_longitude,score_priority_proxy_safe,future_7d_impact,daily_impact_signal_roll_sum_7d,daily_impact_signal_roll_sum_30d,recent_persistence_30d,high_obstruction_share_30d,hist_total_events,hist_confidence_score
4519,junction_BTP051 - Safina Plaza Junction,Shivajinagar,12.981220,77.608711,2961.082431,700.288937,1143.431250,3499.315625,1.000000,0.054182,11313.0,0.953248
4541,junction_BTP082 - KR Market Junction,City Market,12.964379,77.576965,2296.323833,541.886625,717.823688,3083.872563,1.000000,0.033145,9096.0,0.948232
4511,junction_BTP040 - Elite Junction,Upparpet,12.976935,77.576479,1694.189519,618.474875,518.243812,2346.376062,1.000000,0.003519,8693.0,0.947129
4515,junction_BTP044 - Sagar Theatre Junction,Upparpet,12.974472,77.578938,1418.142654,634.721125,319.601813,2154.188313,1.000000,0.020233,8459.0,0.946455
4638,junction_BTP211 - Central Street Junction,Shivajinagar,12.984038,77.603449,1246.698657,112.533188,538.066937,1398.780062,1.000000,0.061947,3990.0,0.920998
4523,junction_BTP057 - Anand Rao Junction,Upparpet,12.979291,77.574870,853.802669,358.336688,354.350688,1063.749250,0.933333,0.025335,3043.0,0.907792
4524,junction_BTP058 - Subbanna Junction,Upparpet,12.978968,77.578605,825.919715,578.488438,209.531250,1241.464375,1.000000,0.008424,4048.0,0.921487
3739,h3_89618920923ffff,HAL Old Airport,12.933571,77.691013,793.903546,45.684000,128.030000,1386.480500,0.633333,0.300493,5575.0,0.844208
4300,h3_8961892e2bbffff,K.R. Pura,13.008209,77.695366,748.479477,382.339750,98.084600,1063.797950,1.000000,0.404609,2490.0,0.903416
3746,h3_89618920963ffff,HAL Old Airport,12.939753,77.695544,671.138091,5.477000,122.517200,921.545950,0.600000,0.926045,1304.0,0.754810


In [10]:
from sklearn.neighbors import BallTree

# =========================
# STATIC NEIGHBOR LOOKUP
# =========================

unit_locations = (
    daily_model[["spatial_unit_id", "unit_latitude", "unit_longitude"]]
    .drop_duplicates("spatial_unit_id")
    .dropna(subset=["unit_latitude", "unit_longitude"])
    .reset_index(drop=True)
)

coords_rad = np.radians(unit_locations[["unit_latitude", "unit_longitude"]].to_numpy())

tree = BallTree(coords_rad, metric="haversine")
radius_rad = NEIGHBOR_RADIUS_KM / EARTH_RADIUS_KM

neighbor_idx_lists = tree.query_radius(coords_rad, r=radius_rad)

unit_ids = unit_locations["spatial_unit_id"].to_numpy()

edges = []

for i, neighbor_idxs in enumerate(neighbor_idx_lists):
    this_unit = unit_ids[i]

    for j in neighbor_idxs:
        if j == i:
            continue

        edges.append((this_unit, unit_ids[j]))

neighbor_edges = pd.DataFrame(
    edges,
    columns=["spatial_unit_id", "neighbor_unit_id"],
)

neighbor_count = (
    neighbor_edges
    .groupby("spatial_unit_id")
    .size()
    .rename("neighbor_count")
    .reset_index()
)

print(f"Neighbor radius: {NEIGHBOR_RADIUS_KM} km")
print(f"Total spatial units: {len(unit_locations)}")
print(f"Units with at least one neighbor: {neighbor_edges['spatial_unit_id'].nunique()}")
print(f"Total directed neighbor edges: {len(neighbor_edges)}")

if len(neighbor_count):
    print(f"Average neighbors among units with neighbors: {neighbor_count['neighbor_count'].mean():.1f}")
    print(f"Maximum neighbors for one unit: {neighbor_count['neighbor_count'].max()}")
else:
    print("No neighbor edges found. Neighbor features will be zero.")

# =========================
# AGGREGATE NEIGHBOR HISTORY FEATURES
# All source columns are already shifted/rolling features,
# so this does not use same-day or future activity.
# =========================

NEIGHBOR_SOURCE_COLS = [
    f"{TARGET_COL}_roll_sum_7d",
    f"{TARGET_COL}_roll_sum_30d",
    "count_roll_sum_7d",
    "count_roll_sum_30d",
]

if neighbor_edges.empty:
    daily_model["neighbor_count"] = 0

    for col in NEIGHBOR_SOURCE_COLS:
        daily_model[f"neighbor_{col}"] = 0.0
        daily_model[f"neighbor_{col}_avg"] = 0.0

else:
    neighbor_source = daily_model[["spatial_unit_id", "date"] + NEIGHBOR_SOURCE_COLS].copy()

    neighbor_panel = neighbor_edges.merge(
        neighbor_source.rename(columns={"spatial_unit_id": "neighbor_unit_id"}),
        on="neighbor_unit_id",
        how="left",
    )

    neighbor_agg = (
        neighbor_panel
        .groupby(["spatial_unit_id", "date"], as_index=False)[NEIGHBOR_SOURCE_COLS]
        .sum()
        .rename(columns={c: f"neighbor_{c}" for c in NEIGHBOR_SOURCE_COLS})
    )

    neighbor_agg = neighbor_agg.merge(neighbor_count, on="spatial_unit_id", how="left")

    daily_model = daily_model.merge(
        neighbor_agg,
        on=["spatial_unit_id", "date"],
        how="left",
    )

    neighbor_sum_cols = [f"neighbor_{c}" for c in NEIGHBOR_SOURCE_COLS]
    neighbor_feature_cols = neighbor_sum_cols + ["neighbor_count"]

    daily_model[neighbor_feature_cols] = daily_model[neighbor_feature_cols].fillna(0)

    for col in NEIGHBOR_SOURCE_COLS:
        daily_model[f"neighbor_{col}_avg"] = np.where(
            daily_model["neighbor_count"] > 0,
            daily_model[f"neighbor_{col}"] / daily_model["neighbor_count"],
            0.0,
        )

neighbor_sum_cols = [f"neighbor_{c}" for c in NEIGHBOR_SOURCE_COLS]
neighbor_avg_cols = [f"neighbor_{c}_avg" for c in NEIGHBOR_SOURCE_COLS]
neighbor_feature_cols = neighbor_sum_cols + neighbor_avg_cols + ["neighbor_count"]

# =========================
# SANITY CHECKS
# =========================

print("\nNeighbor feature columns added:")
print(neighbor_feature_cols)

print("\nShare of unit-days with zero neighbors:")
print((daily_model["neighbor_count"] == 0).mean())

if (daily_model["neighbor_count"] > 0).any():
    print("\nNeighbor feature summary for rows with neighbors:")
    display(
        daily_model.loc[
            daily_model["neighbor_count"] > 0,
            neighbor_feature_cols,
        ].describe()
    )
else:
    print("\nAll neighbor feature values are zero because no neighbors were found.")

Neighbor radius: 0.5 km
Total spatial units: 2156
Units with at least one neighbor: 2037
Total directed neighbor edges: 7276
Average neighbors among units with neighbors: 3.6
Maximum neighbors for one unit: 10

Neighbor feature columns added:
['neighbor_daily_impact_signal_roll_sum_7d', 'neighbor_daily_impact_signal_roll_sum_30d', 'neighbor_count_roll_sum_7d', 'neighbor_count_roll_sum_30d', 'neighbor_daily_impact_signal_roll_sum_7d_avg', 'neighbor_daily_impact_signal_roll_sum_30d_avg', 'neighbor_count_roll_sum_7d_avg', 'neighbor_count_roll_sum_30d_avg', 'neighbor_count']

Share of unit-days with zero neighbors:
0.05519480519480519

Neighbor feature summary for rows with neighbors:


,neighbor_daily_impact_signal_roll_sum_7d,neighbor_daily_impact_signal_roll_sum_30d,neighbor_count_roll_sum_7d,neighbor_count_roll_sum_30d,neighbor_daily_impact_signal_roll_sum_7d_avg,neighbor_daily_impact_signal_roll_sum_30d_avg,neighbor_count_roll_sum_7d_avg,neighbor_count_roll_sum_30d_avg,neighbor_count
count,307587.000000,307587.000000,307587.00000,307587.000000,307587.000000,307587.000000,307587.000000,307587.000000,307587.000000
mean,36.156856,145.636105,22.02822,86.355077,9.139574,36.859685,5.424962,21.302252,3.571919
std,118.269098,455.185629,68.59970,259.752122,34.613735,134.520197,19.155356,73.688427,1.746690
min,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,2.887500,0.00000,2.000000,0.000000,1.100000,0.000000,0.666667,2.000000
50%,3.740000,21.996200,2.00000,12.000000,1.100000,6.520000,0.666667,3.666667,3.000000
75%,24.255000,104.580000,15.00000,60.000000,6.160000,26.345125,3.750000,15.500000,5.000000
max,3304.183375,10719.664937,1609.00000,4483.000000,1622.543688,4450.200625,750.000000,2127.000000,10.000000


In [11]:

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# MODEL FEATURES
# =========================

FEATURES = [
    f"{TARGET_COL}_lag_1d",
    f"{TARGET_COL}_lag_7d",
    f"{TARGET_COL}_lag_14d",

    f"{TARGET_COL}_roll_sum_7d",
    f"{TARGET_COL}_roll_mean_7d",
    f"{TARGET_COL}_roll_sum_14d",
    f"{TARGET_COL}_roll_mean_14d",
    f"{TARGET_COL}_roll_sum_30d",
    f"{TARGET_COL}_roll_mean_30d",

    "count_lag_1d",
    "count_lag_7d",
    "count_lag_14d",
    "count_roll_sum_7d",
    "count_roll_sum_14d",
    "count_roll_sum_30d",

    "recent_persistence_7d",
    "recent_persistence_30d",
    "high_obstruction_share_30d",
    "peak_hour_share_30d",

    "impact_momentum_7_vs_30",
    "count_momentum_7_vs_30",

    "hist_total_events",
    "hist_total_impact",
    "hist_active_days",
    "hist_observed_days",
    "hist_active_day_rate",
    "hist_mean_daily_impact",
    "hist_confidence_score",

    "named_junction_share",
    "unit_latitude",
    "unit_longitude",

    "dayofweek",
    "month",
    "is_weekend_day",

    f"neighbor_{TARGET_COL}_roll_sum_7d",
    f"neighbor_{TARGET_COL}_roll_sum_30d",
    "neighbor_count_roll_sum_7d",
    "neighbor_count_roll_sum_30d",

    f"neighbor_{TARGET_COL}_roll_sum_7d_avg",
    f"neighbor_{TARGET_COL}_roll_sum_30d_avg",
    "neighbor_count_roll_sum_7d_avg",
    "neighbor_count_roll_sum_30d_avg",

    "neighbor_count",
]

FEATURES = [c for c in FEATURES if c in daily_model.columns]

print("Feature count:", len(FEATURES))
print(FEATURES)

# =========================
# MODEL FACTORY
# =========================

def make_model():
    if MODEL_TYPE == "hist_gbdt":
        return HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )

    if MODEL_TYPE == "random_forest":
        return RandomForestRegressor(
            n_estimators=250,
            max_depth=14,
            min_samples_leaf=10,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

    raise ValueError("MODEL_TYPE must be 'hist_gbdt' or 'random_forest'")

# =========================
# WALK-FORWARD TRAIN / EVAL
# =========================

base_pool = daily_model[
    (daily_model["has_full_future_horizon"] == 1)
    & (daily_model["hist_total_events"] >= MIN_HIST_EVENTS_FOR_TRAIN)
].copy()

walkforward_rows = []
walkforward_snapshots = []
fold_models = {}

for val_date_str in VALIDATION_DATES:
    val_date = pd.to_datetime(val_date_str)

    train_df = base_pool[base_pool["date"] < val_date].copy()
    snap = base_pool[base_pool["date"] == val_date].copy()

    if len(train_df) < MIN_ROWS_TO_TRAIN_FOLD or snap.empty:
        print(
            f"Skipping {val_date_str}: "
            f"train_rows={len(train_df)}, valid_rows={len(snap)}"
        )
        continue

    X_train = train_df[FEATURES].fillna(0)
    y_train = train_df[LOG_TARGET]

    X_valid = snap[FEATURES].fillna(0)

    model = make_model()
    model.fit(X_train, y_train)

    fold_models[val_date_str] = model

    snap["pred_log_ml"] = model.predict(X_valid)
    snap["pred_impact_ml"] = np.expm1(snap["pred_log_ml"]).clip(lower=0)

    snap["pred_impact_baseline_recent30"] = snap[f"{TARGET_COL}_roll_sum_30d"]

    snap["pred_impact_baseline_priority"] = (
        (
            0.65 * snap[f"{TARGET_COL}_roll_sum_7d"]
            + 0.35 * snap[f"{TARGET_COL}_roll_sum_30d"]
        )
        * (1 + 0.50 * snap["recent_persistence_30d"])
        * (1 + 0.50 * snap["high_obstruction_share_30d"])
        * (0.50 + 0.50 * snap["hist_confidence_score"])
    )

    fold_score_cols = {
        "ML": "pred_impact_ml",
        "Baseline: recent30": "pred_impact_baseline_recent30",
        "Baseline: priority": "pred_impact_baseline_priority",
    }

    for label, score_col in fold_score_cols.items():
        row = {
            "model": label,
            "eval_date": val_date_str,
            "n_units": len(snap),
            "train_rows": len(train_df),
            "train_units": train_df["spatial_unit_id"].nunique(),
            "spearman": safe_spearman(snap[TARGET], snap[score_col]),
            "rmse_log": np.sqrt(
                mean_squared_error(np.log1p(snap[TARGET]), np.log1p(snap[score_col]))
            ),
            "mae_raw": mean_absolute_error(snap[TARGET], snap[score_col]),
        }

        for k in TOP_K_LIST:
            row[f"recall_at_{k}"] = recall_at_k(snap[TARGET], snap[score_col], k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(
                snap[TARGET].to_numpy(),
                snap[score_col].to_numpy(),
                k,
            )

        walkforward_rows.append(row)

    walkforward_snapshots.append(snap.assign(validation_date=val_date))

if not walkforward_snapshots:
    raise ValueError("No walk-forward folds were run. Check VALIDATION_DATES and data range.")

model_eval = pd.DataFrame(walkforward_rows)
valid_df = pd.concat(walkforward_snapshots, ignore_index=True)

# =========================
# RESULTS
# =========================

print("Walk-forward ML evaluation complete.")
print("Model type:", MODEL_TYPE)
print("Folds run:", list(fold_models.keys()))

print("\nPer-fold validation metrics:")
display(
    model_eval.sort_values(
        ["eval_date", "recall_at_50", "ndcg_at_50", "spearman"],
        ascending=False,
    )
)

walkforward_summary = (
    model_eval
    .groupby("model", as_index=False)
    .mean(numeric_only=True)
    .sort_values(["recall_at_50", "ndcg_at_50", "spearman"], ascending=False)
)

print("\nWalk-forward average comparison:")
display(walkforward_summary)

# =========================
# FEATURE IMPORTANCE / EXPLAINABILITY PROXY
# =========================

last_fold_date = list(fold_models.keys())[-1]
last_model = fold_models[last_fold_date]
last_fold_snap = walkforward_snapshots[-1]
X_valid_last = last_fold_snap[FEATURES].fillna(0)

if MODEL_TYPE == "random_forest":
    feature_importance = pd.DataFrame(
        {
            "feature": FEATURES,
            "importance": last_model.feature_importances_,
        }
    ).sort_values("importance", ascending=False)
else:
    feature_importance = pd.DataFrame(
        {
            "feature": FEATURES,
            "pred_corr_abs": [
                abs(pd.Series(X_valid_last[c]).corr(pd.Series(last_fold_snap["pred_impact_ml"])))
                for c in FEATURES
            ],
        }
    ).sort_values("pred_corr_abs", ascending=False)

print(f"\nFeature importance / prediction-correlation proxy from fold {last_fold_date}:")
display(feature_importance.head(20))

# =========================
# LATEST ML RECOMMENDATIONS
# =========================

latest_date = pd.to_datetime(VALIDATION_DATES[-1])

latest_ml = valid_df[valid_df["date"] == latest_date].copy()
latest_ml = latest_ml.sort_values("pred_impact_ml", ascending=False)

print(f"\nTop ML hotspot recommendations as of {latest_date.date()}:")
display(
    latest_ml.head(25)[
        [
            "spatial_unit_id",
            "police_station_mode",
            "unit_latitude",
            "unit_longitude",
            "pred_impact_ml",
            TARGET,
            "pred_impact_baseline_recent30",
            "pred_impact_baseline_priority",
            f"{TARGET_COL}_roll_sum_7d",
            f"{TARGET_COL}_roll_sum_30d",
            "hist_total_events",
            "hist_confidence_score",
        ]
    ]
)


Feature count: 43
['daily_impact_signal_lag_1d', 'daily_impact_signal_lag_7d', 'daily_impact_signal_lag_14d', 'daily_impact_signal_roll_sum_7d', 'daily_impact_signal_roll_mean_7d', 'daily_impact_signal_roll_sum_14d', 'daily_impact_signal_roll_mean_14d', 'daily_impact_signal_roll_sum_30d', 'daily_impact_signal_roll_mean_30d', 'count_lag_1d', 'count_lag_7d', 'count_lag_14d', 'count_roll_sum_7d', 'count_roll_sum_14d', 'count_roll_sum_30d', 'recent_persistence_7d', 'recent_persistence_30d', 'high_obstruction_share_30d', 'peak_hour_share_30d', 'impact_momentum_7_vs_30', 'count_momentum_7_vs_30', 'hist_total_events', 'hist_total_impact', 'hist_active_days', 'hist_observed_days', 'hist_active_day_rate', 'hist_mean_daily_impact', 'hist_confidence_score', 'named_junction_share', 'unit_latitude', 'unit_longitude', 'dayofweek', 'month', 'is_weekend_day', 'neighbor_daily_impact_signal_roll_sum_7d', 'neighbor_daily_impact_signal_roll_sum_30d', 'neighbor_count_roll_sum_7d', 'neighbor_count_roll_sum_

,model,eval_date,n_units,train_rows,train_units,spearman,rmse_log,mae_raw,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
11,Baseline: priority,2024-04-01,1368,137763,1361,0.633187,1.214265,21.802947,0.7,0.903976,0.72,0.925690,0.70,0.923181,0.66,0.903982
9,ML,2024-04-01,1368,137763,1361,0.694445,0.808026,6.064128,0.6,0.906058,0.76,0.922906,0.68,0.909350,0.65,0.899969
10,Baseline: recent30,2024-04-01,1368,137763,1361,0.615271,1.702009,39.632316,0.7,0.915289,0.68,0.906860,0.64,0.900675,0.64,0.895539
6,ML,2024-03-01,1252,97048,1250,0.692048,0.848582,6.636347,0.7,0.940283,0.68,0.922533,0.74,0.931740,0.66,0.916779
7,Baseline: recent30,2024-03-01,1252,97048,1250,0.615533,1.683880,36.982487,0.7,0.904407,0.68,0.912502,0.68,0.910712,0.62,0.896679
8,Baseline: priority,2024-03-01,1252,97048,1250,0.616263,1.275302,21.616834,0.7,0.939186,0.72,0.926854,0.68,0.906270,0.63,0.909042
5,Baseline: priority,2024-02-01,1127,62736,1122,0.634396,1.543666,39.988316,0.7,0.903958,0.76,0.941770,0.66,0.918283,0.68,0.898585
3,ML,2024-02-01,1127,62736,1122,0.720659,0.793246,6.524604,0.7,0.904296,0.76,0.946913,0.64,0.918512,0.75,0.923345
4,Baseline: recent30,2024-02-01,1127,62736,1122,0.630159,2.196041,77.616606,0.7,0.901931,0.80,0.942680,0.64,0.913442,0.70,0.895714
0,ML,2024-01-01,905,30775,893,0.800275,0.904555,13.701567,0.8,0.919716,0.64,0.929443,0.70,0.925114,0.69,0.921766



Walk-forward average comparison:


,model,n_units,train_rows,train_units,spearman,rmse_log,mae_raw,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
2,ML,1163.0,82080.5,1156.5,0.726857,0.838602,8.231662,0.700,0.917588,0.71,0.930449,0.690,0.921179,0.6875,0.915465
0,Baseline: priority,1163.0,82080.5,1156.5,0.635327,1.389435,31.272985,0.750,0.916456,0.71,0.923926,0.685,0.910458,0.6450,0.898035
1,Baseline: recent30,1163.0,82080.5,1156.5,0.630384,1.925616,58.581012,0.725,0.904902,0.70,0.916457,0.660,0.905064,0.6450,0.892037



Feature importance / prediction-correlation proxy from fold 2024-04-01:


,feature,pred_corr_abs
22,hist_total_impact,0.964848
26,hist_mean_daily_impact,0.964848
7,daily_impact_signal_roll_sum_30d,0.962256
8,daily_impact_signal_roll_mean_30d,0.962256
5,daily_impact_signal_roll_sum_14d,0.960548
6,daily_impact_signal_roll_mean_14d,0.960548
14,count_roll_sum_30d,0.960361
13,count_roll_sum_14d,0.954859
21,hist_total_events,0.953818
4,daily_impact_signal_roll_mean_7d,0.933219



Top ML hotspot recommendations as of 2024-04-01:


,spatial_unit_id,police_station_mode,unit_latitude,unit_longitude,pred_impact_ml,future_7d_impact,pred_impact_baseline_recent30,pred_impact_baseline_priority,daily_impact_signal_roll_sum_7d,daily_impact_signal_roll_sum_30d,hist_total_events,hist_confidence_score
4541,junction_BTP082 - KR Market Junction,City Market,12.964379,77.576965,663.053507,541.886625,3083.872563,2296.323833,717.823688,3083.872563,9096.0,0.948232
4519,junction_BTP051 - Safina Plaza Junction,Shivajinagar,12.981220,77.608711,600.245412,700.288937,3499.315625,2961.082431,1143.431250,3499.315625,11313.0,0.953248
4511,junction_BTP040 - Elite Junction,Upparpet,12.976935,77.576479,578.015789,618.474875,2346.376062,1694.189519,518.243812,2346.376062,8693.0,0.947129
4515,junction_BTP044 - Sagar Theatre Junction,Upparpet,12.974472,77.578938,458.185220,634.721125,2154.188313,1418.142654,319.601813,2154.188313,8459.0,0.946455
4638,junction_BTP211 - Central Street Junction,Shivajinagar,12.984038,77.603449,377.707646,112.533188,1398.780062,1246.698657,538.066937,1398.780062,3990.0,0.920998
4523,junction_BTP057 - Anand Rao Junction,Upparpet,12.979291,77.574870,300.343856,358.336688,1063.749250,853.802669,354.350688,1063.749250,3043.0,0.907792
4524,junction_BTP058 - Subbanna Junction,Upparpet,12.978968,77.578605,252.912180,578.488438,1241.464375,825.919715,209.531250,1241.464375,4048.0,0.921487
4540,"junction_BTP080 - NR Road, SP Road Junction",City Market,12.964332,77.583940,201.254768,197.088500,772.065438,550.910506,191.563875,772.065438,2837.0,0.892871
4501,junction_BTP027 - Modi Bridge Junction,Rajajinagar,12.999177,77.549582,170.332081,226.771375,757.899688,626.958106,256.135937,757.899688,3541.0,0.913351
4516,junction_BTP045 - Danvanthri Road Junction,Upparpet,12.977611,77.574786,149.044574,221.389500,710.838562,524.681223,166.728625,710.838562,2438.0,0.899055


In [20]:

# =========================
# ENSEMBLE WEIGHT SEARCH
# =========================

steps = np.round(np.arange(0, 1 + 1e-9, WEIGHT_STEP), 2)

ENSEMBLE_WEIGHTS = []

for w_ml in steps:
    for w_recent30 in steps:
        w_priority = round(1 - w_ml - w_recent30, 2)

        if w_priority < -1e-9:
            continue

        ENSEMBLE_WEIGHTS.append(
            {
                "ml": round(float(w_ml), 2),
                "recent30": round(float(w_recent30), 2),
                "priority": round(float(max(w_priority, 0.0)), 2),
            }
        )

print(f"Searching {len(ENSEMBLE_WEIGHTS)} ensemble weight combinations.")

# =========================
# BASE SCORE PREP
# =========================

ensemble_df = valid_df.copy()

ensemble_df["score_ml"] = ensemble_df["pred_impact_ml"]
ensemble_df["score_recent30"] = ensemble_df[f"{TARGET_COL}_roll_sum_30d"]

ensemble_df["score_priority"] = (
    (
        0.65 * ensemble_df[f"{TARGET_COL}_roll_sum_7d"]
        + 0.35 * ensemble_df[f"{TARGET_COL}_roll_sum_30d"]
    )
    * (1 + 0.50 * ensemble_df["recent_persistence_30d"])
    * (1 + 0.50 * ensemble_df["high_obstruction_share_30d"])
    * (0.50 + 0.50 * ensemble_df["hist_confidence_score"])
)

def add_rank_percentile(frame, score_col):
    return frame.groupby("date")[score_col].rank(pct=True, ascending=True)

ensemble_df["rank_ml"] = add_rank_percentile(ensemble_df, "score_ml")
ensemble_df["rank_recent30"] = add_rank_percentile(ensemble_df, "score_recent30")
ensemble_df["rank_priority"] = add_rank_percentile(ensemble_df, "score_priority")

# =========================
# CREATE ENSEMBLE CANDIDATES
# =========================

score_candidates = {
    "ML": "score_ml",
    "Recent30 baseline": "score_recent30",
    "Priority baseline": "score_priority",
}

weight_lookup = {}

for i, w in enumerate(ENSEMBLE_WEIGHTS, start=1):
    col = f"score_ensemble_{i}"
    label = f"ML{w['ml']:.2f}_R{w['recent30']:.2f}_P{w['priority']:.2f}"

    ensemble_df[col] = (
        w["ml"] * ensemble_df["rank_ml"]
        + w["recent30"] * ensemble_df["rank_recent30"]
        + w["priority"] * ensemble_df["rank_priority"]
    )

    score_candidates[label] = col
    weight_lookup[label] = w

# =========================
# EVALUATE SCORE CANDIDATES
# =========================

available_eval_dates = sorted(ensemble_df["date"].dropna().unique())

def evaluate_score(frame, score_col, label):
    rows = []

    for eval_date in available_eval_dates:
        snap = frame[frame["date"] == eval_date].copy()

        if snap.empty:
            continue

        row = {
            "score_name": label,
            "eval_date": pd.to_datetime(eval_date).date(),
            "n_units": len(snap),
            "spearman": safe_spearman(snap[TARGET], snap[score_col]),
        }

        for k in TOP_K_LIST:
            row[f"recall_at_{k}"] = recall_at_k(snap[TARGET], snap[score_col], k)
            row[f"ndcg_at_{k}"] = ndcg_at_k(
                snap[TARGET].to_numpy(),
                snap[score_col].to_numpy(),
                k,
            )

        rows.append(row)

    return rows

ensemble_eval_rows = []

for label, col in score_candidates.items():
    ensemble_eval_rows.extend(evaluate_score(ensemble_df, col, label))

ensemble_eval = pd.DataFrame(ensemble_eval_rows)

ensemble_summary = (
    ensemble_eval
    .groupby("score_name", as_index=False)
    .mean(numeric_only=True)
    .sort_values(["recall_at_50", "ndcg_at_50", "spearman"], ascending=False)
)

BEST_FINAL_SCORE_LABEL = ensemble_summary.iloc[0]["score_name"]
BEST_FINAL_SCORE_COL = score_candidates[BEST_FINAL_SCORE_LABEL]
BEST_FINAL_WEIGHTS = weight_lookup.get(BEST_FINAL_SCORE_LABEL, None)

print("Ensemble search complete.")
print("Best final score:", BEST_FINAL_SCORE_LABEL)
print("Best score column:", BEST_FINAL_SCORE_COL)

if BEST_FINAL_WEIGHTS is not None:
    print("Best ensemble weights:", BEST_FINAL_WEIGHTS)

print("\nTop 15 score candidates:")
display(ensemble_summary.head(15))

print("\nPer-date comparison for best score, ML, and baselines:")
comparison_labels = [
    BEST_FINAL_SCORE_LABEL,
    "ML",
    "Recent30 baseline",
    "Priority baseline",
]

display(
    ensemble_eval[ensemble_eval["score_name"].isin(comparison_labels)]
    .sort_values(["eval_date", "recall_at_50", "ndcg_at_50"], ascending=False)
)

# =========================
# FINAL RECOMMENDATIONS FROM LATEST WALK-FORWARD SNAPSHOT
# =========================

LATEST_DECISION_DATE = pd.to_datetime(max(available_eval_dates))

final_recommendations = ensemble_df[
    ensemble_df["date"] == LATEST_DECISION_DATE
].copy()

final_recommendations["final_priority_score"] = final_recommendations[BEST_FINAL_SCORE_COL]

final_recommendations["priority_rank"] = (
    final_recommendations["final_priority_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

final_recommendations["priority_tier"] = pd.cut(
    final_recommendations["priority_rank"],
    bins=[0, 25, 100, 300, np.inf],
    labels=["Critical", "High", "Medium", "Watchlist"],
)

final_recommendations["location_name"] = (
    final_recommendations["spatial_unit_id"]
    .astype(str)
    .str.replace("junction_", "", regex=False)
)

final_recommendations["location_type"] = np.where(
    final_recommendations["spatial_unit_id"].astype(str).str.startswith("junction_"),
    "Named junction",
    "H3/grid hotspot cell",
)

final_recommendations["recommended_action"] = final_recommendations["priority_tier"].map(
    {
        "Critical": "Daily focused enforcement",
        "High": "Scheduled enforcement patrol",
        "Medium": "Periodic monitoring",
        "Watchlist": "Watchlist",
    }
)

final_recommendations["recommended_time_window"] = "Peak parking-risk windows"
final_recommendations["priority_reason"] = "High rank from ML, recent history, and operational priority ensemble"

final_recommendations = final_recommendations.sort_values("priority_rank")

final_hotspots = final_recommendations[
    [
        "priority_rank",
        "priority_tier",
        "spatial_unit_id",
        "location_name",
        "location_type",
        "police_station_mode",
        "unit_latitude",
        "unit_longitude",
        "final_priority_score",
        "score_ml",
        "score_recent30",
        "score_priority",
        f"{TARGET_COL}_roll_sum_7d",
        f"{TARGET_COL}_roll_sum_30d",
        "recent_persistence_30d",
        "high_obstruction_share_30d",
        "hist_total_events",
        "hist_confidence_score",
        "recommended_action",
        "recommended_time_window",
        "priority_reason",
    ]
].copy()

final_hotspots.to_csv(FINAL_HOTSPOTS_PATH, index=False)

print(f"\nFinal recommendations as of {LATEST_DECISION_DATE.date()}:")
print("Saved:", FINAL_HOTSPOTS_PATH)

display(final_hotspots.head(20))


Searching 231 ensemble weight combinations.
Ensemble search complete.
Best final score: ML0.70_R0.15_P0.15
Best score column: score_ensemble_207
Best ensemble weights: {'ml': 0.7, 'recent30': 0.15, 'priority': 0.15}

Top 15 score candidates:


,score_name,n_units,spearman,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
207,ML0.70_R0.15_P0.15,1163.0,0.712895,0.725,0.918190,0.72,0.931091,0.695,0.923767,0.6800,0.914381
208,ML0.70_R0.20_P0.10,1163.0,0.712852,0.725,0.918092,0.72,0.930964,0.695,0.923716,0.6800,0.914348
181,ML0.55_R0.20_P0.25,1163.0,0.701203,0.725,0.922589,0.72,0.930898,0.695,0.923404,0.6700,0.912842
214,ML0.75_R0.15_P0.10,1163.0,0.716039,0.725,0.917357,0.72,0.930438,0.695,0.922842,0.6825,0.915582
225,ML0.85_R0.15_P0.00,1163.0,0.721519,0.725,0.917357,0.72,0.930016,0.695,0.922754,0.6825,0.914569
213,ML0.75_R0.10_P0.15,1163.0,0.716108,0.725,0.917357,0.72,0.930255,0.695,0.922670,0.6875,0.916311
219,ML0.80_R0.10_P0.10,1163.0,0.718996,0.725,0.918968,0.72,0.930213,0.695,0.922624,0.6825,0.915580
206,ML0.70_R0.10_P0.20,1163.0,0.712874,0.725,0.917455,0.72,0.930247,0.695,0.922592,0.6825,0.914226
205,ML0.70_R0.05_P0.25,1163.0,0.712805,0.725,0.917386,0.72,0.930068,0.695,0.922313,0.6825,0.914077
171,ML0.50_R0.25_P0.25,1163.0,0.696700,0.725,0.914369,0.71,0.922541,0.695,0.917433,0.6725,0.907559



Per-date comparison for best score, ML, and baselines:


,score_name,eval_date,n_units,spearman,recall_at_10,ndcg_at_10,recall_at_25,ndcg_at_25,recall_at_50,ndcg_at_50,recall_at_100,ndcg_at_100
11,Priority baseline,2024-04-01,1368,0.633187,0.7,0.903976,0.72,0.925690,0.70,0.923181,0.66,0.903982
3,ML,2024-04-01,1368,0.694445,0.6,0.906058,0.76,0.922906,0.68,0.909350,0.65,0.899969
839,ML0.70_R0.15_P0.15,2024-04-01,1368,0.686170,0.7,0.908983,0.76,0.918043,0.68,0.908440,0.67,0.902382
7,Recent30 baseline,2024-04-01,1368,0.615271,0.7,0.915289,0.68,0.906860,0.64,0.900675,0.64,0.895539
2,ML,2024-03-01,1252,0.692048,0.7,0.940283,0.68,0.922533,0.74,0.931740,0.66,0.916779
838,ML0.70_R0.15_P0.15,2024-03-01,1252,0.680267,0.7,0.938009,0.72,0.930182,0.72,0.931218,0.64,0.917487
6,Recent30 baseline,2024-03-01,1252,0.615533,0.7,0.904407,0.68,0.912502,0.68,0.910712,0.62,0.896679
10,Priority baseline,2024-03-01,1252,0.616263,0.7,0.939186,0.72,0.926854,0.68,0.906270,0.63,0.909042
837,ML0.70_R0.15_P0.15,2024-02-01,1127,0.709689,0.7,0.905661,0.76,0.947846,0.68,0.927468,0.73,0.917603
9,Priority baseline,2024-02-01,1127,0.634396,0.7,0.903958,0.76,0.941770,0.66,0.918283,0.68,0.898585



Final recommendations as of 2024-04-01:
Saved: outputs\final_hotspots.csv


,priority_rank,priority_tier,spatial_unit_id,location_name,location_type,police_station_mode,unit_latitude,unit_longitude,final_priority_score,score_ml,...,score_priority,daily_impact_signal_roll_sum_7d,daily_impact_signal_roll_sum_30d,recent_persistence_30d,high_obstruction_share_30d,hist_total_events,hist_confidence_score,recommended_action,recommended_time_window,priority_reason
4541,1,Critical,junction_BTP082 - KR Market Junction,BTP082 - KR Market Junction,Named junction,City Market,12.964379,77.576965,0.999781,663.053507,...,2296.323833,717.823688,3083.872563,1.000000,0.033145,9096.0,0.948232,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4519,2,Critical,junction_BTP051 - Safina Plaza Junction,BTP051 - Safina Plaza Junction,Named junction,Shivajinagar,12.981220,77.608711,0.999488,600.245412,...,2961.082431,1143.431250,3499.315625,1.000000,0.054182,11313.0,0.953248,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4511,3,Critical,junction_BTP040 - Elite Junction,BTP040 - Elite Junction,Named junction,Upparpet,12.976935,77.576479,0.998538,578.015789,...,1694.189519,518.243812,2346.376062,1.000000,0.003519,8693.0,0.947129,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4515,4,Critical,junction_BTP044 - Sagar Theatre Junction,BTP044 - Sagar Theatre Junction,Named junction,Upparpet,12.974472,77.578938,0.997807,458.185220,...,1418.142654,319.601813,2154.188313,1.000000,0.020233,8459.0,0.946455,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4638,5,Critical,junction_BTP211 - Central Street Junction,BTP211 - Central Street Junction,Named junction,Shivajinagar,12.984038,77.603449,0.997076,377.707646,...,1246.698657,538.066937,1398.780062,1.000000,0.061947,3990.0,0.920998,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4523,6,Critical,junction_BTP057 - Anand Rao Junction,BTP057 - Anand Rao Junction,Named junction,Upparpet,12.979291,77.574870,0.996016,300.343856,...,853.802669,354.350688,1063.749250,0.933333,0.025335,3043.0,0.907792,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4524,7,Critical,junction_BTP058 - Subbanna Junction,BTP058 - Subbanna Junction,Named junction,Upparpet,12.978968,77.578605,0.995614,252.912180,...,825.919715,209.531250,1241.464375,1.000000,0.008424,4048.0,0.921487,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4540,8,Critical,"junction_BTP080 - NR Road, SP Road Junction","BTP080 - NR Road, SP Road Junction",Named junction,City Market,12.964332,77.583940,0.993458,201.254768,...,550.910506,191.563875,772.065438,0.900000,0.033956,2837.0,0.892871,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4501,9,Critical,junction_BTP027 - Modi Bridge Junction,BTP027 - Modi Bridge Junction,Named junction,Rajajinagar,12.999177,77.549582,0.993275,170.332081,...,626.958106,256.135937,757.899688,0.966667,0.046584,3541.0,0.913351,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."
4485,10,Critical,"junction_BTP001 - 10th Cross, Dr. Rajkumar Road","BTP001 - 10th Cross, Dr. Rajkumar Road",Named junction,Malleshwaram,13.008264,77.554523,0.992580,148.498362,...,619.528259,148.741563,987.838813,0.966667,0.004380,2274.0,0.883919,Daily focused enforcement,Peak parking-risk windows,"High rank from ML, recent history, and operati..."


In [45]:
# =========================
# CLEAN SUBMISSION MAP
# =========================

try:
    import folium
    from folium.plugins import HeatMap
except ImportError:
    raise ImportError("Folium is required. Install with: !pip install folium")

import html

TOP_N_MAP = 120
TOP_N_LABELS = 10
MAP_HTML_PATH = OUTPUT_DIR / "parking_hotspot_map.html"

if "final_hotspots" not in globals():
    raise NameError("Run Cell 10 and Cell 11 first. Expected `final_hotspots`.")

map_df = final_hotspots.copy()
map_df = map_df.dropna(subset=["unit_latitude", "unit_longitude"])
map_df = map_df.sort_values("priority_rank").head(TOP_N_MAP).copy()

if map_df.empty:
    raise ValueError("No valid latitude/longitude rows available for mapping.")

def clean_text(value, fallback="Not available"):
    if pd.isna(value):
        return fallback
    text = str(value).strip()
    return text if text and text.lower() not in {"nan", "none"} else fallback

def esc(value):
    return html.escape(clean_text(value))

def readable_location_name(row):
    spatial_id = clean_text(row.get("spatial_unit_id"), "")

    if spatial_id.startswith("junction_"):
        return spatial_id.replace("junction_", "").strip()

    station = clean_text(row.get("police_station_mode"), "Nearby")
    rank = int(row.get("priority_rank", 0))
    return f"{station} hotspot zone #{rank}"

def readable_location_type(row):
    spatial_id = clean_text(row.get("spatial_unit_id"), "")

    if spatial_id.startswith("junction_"):
        return "Named traffic junction"
    if spatial_id.startswith("h3_"):
        return "Local hotspot cell"
    if spatial_id.startswith("grid_"):
        return "Grid hotspot cell"
    return "Spatial hotspot"

def fmt_num(x, digits=2):
    if pd.isna(x):
        return "NA"
    return f"{float(x):.{digits}f}"

def fmt_pct(x):
    if pd.isna(x):
        return "NA"
    return f"{100 * float(x):.1f}%"
def fmt_priority(x):
    if pd.isna(x):
        return "NA"
    return f"{float(x):.3f} / 1.000"

def priority_interpretation(score):
    if pd.isna(score):
        return "Priority unavailable"
    score = float(score)

    if score >= 0.95:
        return "Very high enforcement priority"
    if score >= 0.85:
        return "High enforcement priority"
    if score >= 0.70:
        return "Moderate enforcement priority"
    return "Monitoring priority"

def evidence_summary(row):
    parts = []

    if float(row.get("recent_persistence_30d", 0) or 0) >= 0.40:
        parts.append("frequent repeat activity")

    if float(row.get("high_obstruction_share_30d", 0) or 0) >= 0.30:
        parts.append("high obstruction pattern")

    if float(row.get("score_recent30", 0) or 0) > float(row.get(f"{TARGET_COL}_roll_sum_7d", 0) or 0):
        parts.append("strong 30-day history")

    if float(row.get("hist_confidence_score", 0) or 0) >= 0.60:
        parts.append("reliable historical signal")

    return ", ".join(parts) if parts else "mixed model and historical signal"

map_df["display_location_name"] = map_df.apply(readable_location_name, axis=1)
map_df["display_location_type"] = map_df.apply(readable_location_type, axis=1)

center_lat = map_df["unit_latitude"].median()
center_lon = map_df["unit_longitude"].median()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles="CartoDB positron",
    control_scale=True,
)

shared_css = """
<style>
.hotspot-panel {
    position: fixed;
    top: 18px;
    right: 18px;
    z-index: 9999;
    width: 305px;
    background: white;
    border: 1px solid #c9ced6;
    border-radius: 8px;
    padding: 12px 14px;
    font-family: Arial, sans-serif;
    font-size: 12px;
    color: #172033;
    box-shadow: 0 3px 14px rgba(0,0,0,0.22);
}
.hotspot-panel h3 {
    margin: 0 0 8px 0;
    font-size: 16px;
}
.hotspot-panel hr {
    border: none;
    border-top: 1px solid #e3e6ea;
    margin: 9px 0;
}
.tier-row {
    display: flex;
    align-items: center;
    gap: 7px;
    margin: 4px 0;
}
.tier-chip {
    width: 12px;
    height: 12px;
    border-radius: 2px;
}
.chip-critical { background: #d7191c; }
.chip-high { background: #fdae61; }
.chip-medium { background: #ffd92f; }
.chip-watchlist { background: #1a9850; }

.hotspot-popup.critical .head { background: #d7191c; }
.hotspot-popup.high .head { background: #fdae61; color: #172033; }
.hotspot-popup.medium .head { background: #ffd92f; color: #172033; }
.hotspot-popup.watchlist .head { background: #1a9850; }

.hotspot-popup {
    width: 335px;
    font-family: Arial, sans-serif;
    color: #172033;
}
.hotspot-popup .head {
    color: white;
    padding: 9px 11px;
    font-size: 15px;
    font-weight: 700;
}

.hotspot-popup .body {
    padding: 9px 11px 11px 11px;
}
.hotspot-popup h4 {
    margin: 0 0 4px 0;
    font-size: 14px;
}
.hotspot-popup .subtle {
    color: #5b6472;
    font-size: 11px;
    margin-bottom: 8px;
}
.hint { color: #64748b; font-size: 10px; }
.hotspot-popup table {
    width: 100%;
    border-collapse: collapse;
    font-size: 11px;
}
.hotspot-popup td {
    border-top: 1px solid #edf0f3;
    padding: 4px 2px;
    vertical-align: top;
}
.hotspot-popup td:first-child {
    width: 46%;
    color: #475160;
    font-weight: 700;
}
.hotspot-popup .action {
    margin-top: 8px;
    padding: 7px 8px;
    background: #f4f6f8;
    border-left: 3px solid #2f67a6;
    font-size: 11px;
}
.rank-label {
    font-size: 9px;
    font-weight: 650;
    color: white;
    background: #73110d;
    border: 0.7 px solid white;
    border-radius: 9px;
    width: 22px;
    height: 17px;
    text-align: center;
    line-height: 18px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.35);
}
</style>
"""

m.get_root().header.add_child(folium.Element(shared_css))

tier_color = {
    "Critical": "#d7191c",   # red
    "High": "#fdae61",       # orange
    "Medium": "#ffd92f",     # yellow
    "Watchlist": "#1a9850",  # green
}
tier_border = {
    "Critical": "#8b0000",
    "High": "#b35806",
    "Medium": "#b59b00",
    "Watchlist": "#006837",
}
tier_css = {
    "Critical": "critical",
    "High": "high",
    "Medium": "medium",
    "Watchlist": "watchlist",
}

def marker_radius(rank):
    if rank <= 10:
        return 12
    if rank <= 25:
        return 10
    if rank <= 100:
        return 8
    return 6

heat_df = map_df.copy()
heat_df["heat_weight"] = heat_df["final_priority_score"].rank(pct=True)

HeatMap(
    heat_df[["unit_latitude", "unit_longitude", "heat_weight"]].values.tolist(),
    radius=24,
    blur=18,
    min_opacity=0.22,
    max_zoom=14,
).add_to(m)

for _, row in map_df.iterrows():
    rank = int(row["priority_rank"])
    tier = clean_text(row["priority_tier"], "Watchlist")

    color = tier_color.get(tier, "#8c8c8c")
    border = tier_border.get(tier, "#4d4d4d")
    css_tier = tier_css.get(tier, "watchlist")

    popup_html = f"""
    <div class="hotspot-popup {css_tier}">
        <div class="head">Rank #{rank} | {esc(tier)}</div>
        <div class="body">
            <h4>{esc(row["display_location_name"])}</h4>
            <div class="subtle">{esc(row["display_location_type"])}</div>

            <table>
                <tr><td>Police station</td><td>{esc(row.get("police_station_mode"))}</td></tr>
                <tr><td>Coordinates</td><td>{fmt_num(row["unit_latitude"], 5)}, {fmt_num(row["unit_longitude"], 5)}</td></tr>
                <tr><td>Priority score</td><td>{fmt_priority(row.get("final_priority_score"))}<br><span class="hint">{esc(priority_interpretation(row.get("final_priority_score")))}</span></td></tr>
                <tr><td>Predicted next-7-day parking risk</td><td>{fmt_num(row.get("score_ml"), 2)} next-7-day impact points</td></tr>
                <tr><td>Recent 30-day activity</td><td>{fmt_num(row.get("score_recent30"), 2)} impact points </td></tr>
                <tr><td>Historical risk index</td><td>{fmt_num(row.get("score_priority"), 2)} (persistence + obstruction + confidence)</td></tr>
                <tr><td>Activity (last 7 days)</td><td>{fmt_num(row.get(f"{TARGET_COL}_roll_sum_7d"), 2)}</td></tr>
                <tr><td>Activity (last 30 days)</td><td>{fmt_num(row.get(f"{TARGET_COL}_roll_sum_30d"), 2)}</td></tr>
                <tr><td>Repeat activity (30 days)</td><td>{fmt_pct(row.get("recent_persistence_30d"))}</td></tr>
                <tr><td>Severe obstruction rate</td><td>{fmt_pct(row.get("high_obstruction_share_30d"))}</td></tr>
                <tr><td>Total historical incidents</td><td>{fmt_num(row.get("hist_total_events"), 0)}</td></tr>
                <tr><td>Confidence</td><td>{fmt_pct(row.get("hist_confidence_score"))}</td></tr>
                <tr><td>Location ID</td><td>{esc(row.get("spatial_unit_id"))}</td></tr>
            </table>

            <div class="action">
                <b>Recommended action:</b> {esc(row.get("recommended_action"))}<br>
                <b>Risk Window:</b> {esc(row.get("recommended_time_window"))}<br>
                <b>Key Evidence:</b> {esc(evidence_summary(row))}
            div>
        </div>
    </div>
    """

    tooltip = (
        f"#{rank} | {tier} | "
        f"{clean_text(row.get('display_location_name'))} | "
        f"Score {fmt_num(row.get('final_priority_score'), 3)}"
    )

    folium.CircleMarker(
        location=[row["unit_latitude"], row["unit_longitude"]],
        radius=marker_radius(rank),
        color=border,
        weight=2.2,
        fill=True,
        fill_color=color,
        fill_opacity=0.68,
        popup=folium.Popup(popup_html, max_width=380),
        tooltip=tooltip,
    ).add_to(m)

    if rank <= TOP_N_LABELS:
        folium.Marker(
            location=[row["unit_latitude"], row["unit_longitude"]],
            icon=folium.DivIcon(html=f'<div class="rank-label">{rank}</div>'),
        ).add_to(m)

tier_counts = (
    map_df["priority_tier"]
    .value_counts()
    .reindex(["Critical", "High", "Medium", "Watchlist"])
    .fillna(0)
    .astype(int)
)

top_station = map_df["police_station_mode"].value_counts().head(1)
top_station_text = (
    f"{top_station.index[0]} ({int(top_station.iloc[0])})"
    if len(top_station)
    else "NA"
)

best_score_text = globals().get("BEST_FINAL_SCORE_LABEL", "Final ensemble score")
decision_date_text = str(globals().get("LATEST_DECISION_DATE", "Latest available date"))[:10]

panel_html = f"""
<div class="hotspot-panel">
    <h3>Illegal Parking Hotspot Priority Map</h3>

    <b>Decision date:</b> {decision_date_text}<br>
    <b>Mapped hotspots:</b> Top {len(map_df)}<br>
    <b>Model weights:</b> ML 70% • Recent 15% • Priority 15%<br>
    <b>Top police station:</b> {esc(top_station_text)}

    <hr>

    <b>Enforcement priority</b>
    <div class="tier-row"><span class="tier-chip chip-critical"></span>Critical: rank 1-25 ({tier_counts.get("Critical", 0)})</div>
    <div class="tier-row"><span class="tier-chip chip-high"></span>High: rank 26-100 ({tier_counts.get("High", 0)})</div>
    <div class="tier-row"><span class="tier-chip chip-medium"></span>Medium: rank 101-300 ({tier_counts.get("Medium", 0)})</div>
    <div class="tier-row"><span class="tier-chip chip-watchlist"></span>Watchlist: rank 301+ ({tier_counts.get("Watchlist", 0)})</div>
    <hr>

    <b>How priority is calculated</b><br>
    Priority Score (0–1) =
    70% ML Forecast +
    15% Recent Activity +
    15% Historical Operational Signals

    Higher score = Higher enforcement priority.

    <hr>

    <b>Map guide</b><br>

    Marker color = Enforcement priority<br>
    Blue glow = Predicted parking-risk density

    <hr>

    Click any hotspot to view
    • Predicted risk
    • Historical evidence
    • Confidence
    • Recommended enforcement action
</div>
"""

m.get_root().html.add_child(folium.Element(panel_html))

bounds = map_df[["unit_latitude", "unit_longitude"]].values.tolist()
m.fit_bounds(bounds, padding=(30, 30))

m.save(MAP_HTML_PATH)

print("Saved clean interactive map to:", MAP_HTML_PATH)
print("Mapped rows:", len(map_df))


Saved clean interactive map to: outputs\parking_hotspot_map.html
Mapped rows: 120
